In [43]:
# ============================================================
# 023_daily_scanner_and_incremental_ingest
# ============================================================
#
# Overview
# ----------------
# Daily, incremental paper scanner that detects new papers since last run,
# classifies relevance to existing Research Questions using LLM assistance,
# persists results to Notion (Papers DB + System Config DB), and produces a
# compact daily summary for human review.
#
# Key design goals:
#   - Re-runnable / idempotent (safe to run multiple times)
#   - State-aware (incremental scan from Notion-stored cursor/timestamp)
#   - Schema-robust (detects Notion properties dynamically; supports Notion Data Source API)
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - Notion Research Questions DB (RQs; optional Clusters/Gaps)
#   - Notion Papers DB (existing papers for deduplication + upsert target)
#   - Notion System Config DB (stores Last Run timestamp; minimal schema supported)
#   - Optional: Google Drive folder (PDF ingestion, read-only; may fail gracefully)
#   - OpenAlex API (incremental retrieval since last run)
#   - arXiv API (incremental retrieval since last run)
#
# Outputs:
#   - Updated Notion Papers DB:
#       - New/updated records (OpenAI-enriched fields optional)
#       - Status set (new/pending/excluded depending on your DB schema)
#       - Relations (Papers ↔ RQs/Clusters/Gaps) when available
#   - Updated Notion System Config DB:
#       - A single config page "Daily Scanner Config" updated with Last Run timestamp
#       - If required properties (Name/title, Last Run/date) are missing, they are created
#         via Notion Data Source schema update
#   - Daily review summary artifact (markdown report)
#
# Structure
# ----------------
# Cell 01: Environment setup and imports
# Cell 02: Load configuration + last run state from Notion System Config (Data Source compatible)
# Cell 03: Fetch existing papers for deduplication (Papers DB)
# Cell 04: Fetch Research Questions / Clusters / Gaps (if available)
# Cell 05: Scan OpenAlex for new papers since last run (theme search optional)
# Cell 06: Scan arXiv for new papers since last run (theme search optional)
# Cell 07: Optional Google Drive PDF ingestion (graceful failure)
# Cell 08: Deduplicate and merge papers across sources
# Cell 09: LLM classification against Research Questions (+ optional Clusters/Gaps)
# Cell 10: Upsert to Notion Papers DB
#          - Default: only papers with >=1 RQ match are upserted (configurable)
#          - OpenAI enrichment can populate:
#              Name (English), Authors & Year (English), Core Idea (Japanese),
#              Tags (English), PDF Link (URL)
# Cell 11: Update Notion System Config "Last Run" (Data Source API + auto-create missing props)
# Cell 12: Generate daily review summary artifact
# Cell 13: Display summary + next actions for human review
#
# Notes / Assumptions
# ----------------
# - Notion API may return empty "properties" for /databases/{id} depending on workspace;
#   this notebook supports Notion Data Source API by resolving data_source_id and reading schema there.
# - Papers deduplication keys: DOI / OpenAlex ID / arXiv ID + title fallback.
# - Status taxonomy: new / pending / excluded (actual property type may be select/status/rich_text).
# - Relations are created only if the target relation properties exist in the DB schema.
# - Google Drive ingestion is optional; invalid scopes/credentials should not break the run.
# - All automated judgments must be inspectable (store rationale / tags / core_idea).
# - Human review remains the final gate.

In [5]:
# ============================================================
# Cell 01 — Environment setup and imports
# ============================================================
# Overview:
#   Loads environment variables, configures LLM runtime, and imports
#   all shared libraries needed across the notebook for Notion API,
#   OpenAlex/arXiv scanning, Google Drive integration, and LLM calls.
#
# Inputs / Outputs:
#   Inputs: env.txt (environment variables)
#   Outputs: Configured runtime variables, imported modules
#
# Notes:
#   - All API keys loaded from env.txt
#   - LLM provider and model are fixed for this notebook
#   - Imports organized by category for clarity
#

# --- Mandatory env loading ---
from dotenv import load_dotenv
load_dotenv('env.txt')

# --- Runtime LLM configuration (given / assumed) ---
llm_provider = 'OpenAI'
llm_model = 'gpt-4o-mini'
llm_temperature = 0.0

# --- Standard library imports ---
import os
import json
import re
import time
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
from typing import Dict, List, Optional, Set, Any, Tuple
from collections import defaultdict, Counter

# --- Third-party imports ---
import requests
from notion_client import Client
from openai import OpenAI

# --- Initialize API clients ---
notion_token = os.getenv('NOTION_TOKEN')
notion = Client(auth=notion_token)

openai_api_key = os.getenv('OPENAI_API_KEY')
openai_client = OpenAI(api_key=openai_api_key)

openalex_email = os.getenv('OPENALEX_EMAIL', 'user@example.com')

# --- Notion database IDs (to be loaded or configured) ---
# These will be either hardcoded or loaded from System Config in Cell 02
NOTION_LIT_DB_ID = os.getenv('NOTION_LIT_DB_ID', '')
NOTION_RQ_DB_ID = os.getenv('NOTION_RQ_DB_ID', '')
NOTION_CLUSTERS_DB_ID = os.getenv('NOTION_CLUSTERS_DB_ID', '')
NOTION_GAPS_DB_ID = os.getenv('NOTION_GAPS_DB_ID', '')
NOTION_SYSTEM_CONFIG_DB_ID = os.getenv('NOTION_SYSTEM_CONFIG_DB_ID', '')

# --- Optional Google Drive configuration ---
# ============================================================
# Google Drive configuration (PDF ingestion)
# ============================================================

# OAuth credential paths (Desktop App flow)
GOOGLE_OAUTH_CLIENT_SECRET_JSON = os.getenv("GOOGLE_OAUTH_CLIENT_SECRET_JSON")
GOOGLE_TOKEN_JSON = os.getenv("GOOGLE_TOKEN_JSON")

# Target Google Drive folder containing PDF papers
GDRIVE_FOLDER_ID = os.getenv("DRIVE_FOLDER_ID")

# Read-only scope is sufficient for PDF ingestion
GOOGLE_DRIVE_SCOPES = [
    "https://www.googleapis.com/auth/drive.readonly"
]

# Enable only when folder ID is provided
GDRIVE_ENABLED = bool(GDRIVE_FOLDER_ID)

if GDRIVE_ENABLED:
    if not GOOGLE_OAUTH_CLIENT_SECRET_JSON:
        raise ValueError("GOOGLE_OAUTH_CLIENT_SECRET_JSON not found in env.txt")

    if not GOOGLE_TOKEN_JSON:
        raise ValueError("GOOGLE_TOKEN_JSON not found in env.txt")

    if not os.path.exists(GOOGLE_OAUTH_CLIENT_SECRET_JSON):
        raise FileNotFoundError(
            f"Client secret json not found: {GOOGLE_OAUTH_CLIENT_SECRET_JSON}"
        )

log_info(f"Google Drive ingestion enabled: {GDRIVE_ENABLED}")


# --- Logging and display helpers ---
def log_info(message: str) -> None:
    """Simple logging helper for notebook cells."""
    print(f"[INFO {datetime.now(timezone.utc).isoformat()}] {message}")

def log_error(message: str) -> None:
    """Simple error logging helper."""
    print(f"[ERROR {datetime.now(timezone.utc).isoformat()}] {message}")

def log_warning(message: str) -> None:
    """Simple warning logging helper."""
    print(f"[WARN {datetime.now(timezone.utc).isoformat()}] {message}")

log_info("Environment setup complete")
log_info(f"LLM: {llm_provider} / {llm_model} @ temp={llm_temperature}")
log_info(f"Notion DBs configured: Papers={bool(NOTION_LIT_DB_ID)}, RQ={bool(NOTION_RQ_DB_ID)}")
log_info(f"Google Drive ingestion enabled: {GDRIVE_ENABLED}")


[INFO 2026-01-19T05:07:52.400877+00:00] Google Drive ingestion enabled: True
[INFO 2026-01-19T05:07:52.401987+00:00] Environment setup complete
[INFO 2026-01-19T05:07:52.402084+00:00] LLM: OpenAI / gpt-4o-mini @ temp=0.0
[INFO 2026-01-19T05:07:52.402146+00:00] Notion DBs configured: Papers=True, RQ=True
[INFO 2026-01-19T05:07:52.402198+00:00] Google Drive ingestion enabled: True


In [10]:
# ============================================================
# Cell 02 — Load configuration and last run state from Notion
# ============================================================
# Overview:
#   Queries the Notion System Config database to retrieve the last run
#   timestamp/cursor for incremental scanning. If no prior run exists,
#   defaults to scanning the last 7 days. Also validates all required
#   database IDs are present.
#
# Inputs / Outputs:
#   Inputs: NOTION_SYSTEM_CONFIG_DB_ID, Notion API client
#   Outputs: last_run_timestamp (datetime), config_page_id (str)
#
# Notes:
#   - System Config DB expected to have a single page with Title "Daily Scanner Config"
#   - Last run timestamp stored in a Date property named "Last Run"
#   - If no config page exists, one will be created in Cell 11
#   - Default lookback period: 7 days
#

# --- Validate required database IDs ---
required_dbs = {
    'Papers': NOTION_LIT_DB_ID,
    'Research Questions': NOTION_RQ_DB_ID,
    'Clusters': NOTION_CLUSTERS_DB_ID,
    'Gaps': NOTION_GAPS_DB_ID,
    'System Config': NOTION_SYSTEM_CONFIG_DB_ID
}

missing_dbs = [name for name, db_id in required_dbs.items() if not db_id]
if missing_dbs:
    log_error(f"Missing required database IDs: {', '.join(missing_dbs)}")
    log_error("Please configure environment variables or update Cell 01")
    raise ValueError(f"Missing database IDs: {missing_dbs}")

log_info("All required database IDs present")

# --- Query System Config database for last run state ---
config_page_id = None
last_run_timestamp = None
default_lookback_days = 7

try:
    notion_version = os.getenv("NOTION_VERSION", "2022-06-28")
    url = f"https://api.notion.com/v1/databases/{NOTION_SYSTEM_CONFIG_DB_ID}/query"

    headers = {
        "Authorization": f"Bearer {notion_token}",
        "Notion-Version": notion_version,
        "Content-Type": "application/json",
    }

    payload = {
        "filter": {
            "property": "Name",
            "title": {"equals": "Daily Scanner Config"}
        }
    }

    r = requests.post(url, headers=headers, json=payload, timeout=30)
    r.raise_for_status()
    response = r.json()

    if response.get("results"):
        config_page = response["results"][0]
        config_page_id = config_page["id"]

        last_run_prop = config_page["properties"].get("Last Run", {})
        if last_run_prop.get("date") and last_run_prop["date"].get("start"):
            last_run_str = last_run_prop["date"]["start"]
            last_run_timestamp = datetime.fromisoformat(last_run_str.replace("Z", "+00:00"))
            log_info(f"Found existing config page: {config_page_id}")
            log_info(f"Last run timestamp: {last_run_timestamp.isoformat()}")
        else:
            log_info("Config page exists but no last run timestamp found")
    else:
        log_info("No existing config page found (will be created in Cell 11)")

except Exception as e:
    log_error(f"Error querying System Config database: {str(e)}")
    log_info("Continuing with default lookback period")


# --- Set default if no last run found ---
if last_run_timestamp is None:
    last_run_timestamp = datetime.now(timezone.utc) - timedelta(days=default_lookback_days)
    log_info(f"No last run found; defaulting to {default_lookback_days} days ago")
    log_info(f"Default last run timestamp: {last_run_timestamp.isoformat()}")

# --- Calculate time since last run ---
current_timestamp = datetime.now(timezone.utc)
time_since_last_run = current_timestamp - last_run_timestamp
log_info(f"Time since last run: {time_since_last_run}")
log_info(f"Scanning for papers published after: {last_run_timestamp.isoformat()}")

# --- Store for use in subsequent cells ---
config_state = {
    'config_page_id': config_page_id,
    'last_run_timestamp': last_run_timestamp,
    'current_timestamp': current_timestamp,
    'is_first_run': config_page_id is None
}

log_info(f"Configuration loaded successfully (first run: {config_state['is_first_run']})")


[INFO 2026-01-19T05:13:08.785548+00:00] All required database IDs present
[INFO 2026-01-19T05:13:09.426291+00:00] No existing config page found (will be created in Cell 11)
[INFO 2026-01-19T05:13:09.426622+00:00] No last run found; defaulting to 7 days ago
[INFO 2026-01-19T05:13:09.426651+00:00] Default last run timestamp: 2026-01-12T05:13:09.426614+00:00
[INFO 2026-01-19T05:13:09.426758+00:00] Time since last run: 7 days, 0:00:00.000086
[INFO 2026-01-19T05:13:09.426823+00:00] Scanning for papers published after: 2026-01-12T05:13:09.426614+00:00
[INFO 2026-01-19T05:13:09.426915+00:00] Configuration loaded successfully (first run: True)


In [12]:
# ============================================================
# Cell 03 — Fetch existing papers for deduplication
# ============================================================
# Overview:
#   Queries the Notion Papers database to retrieve all existing papers
#   for deduplication purposes. Builds lookup indices by DOI, OpenAlex ID,
#   arXiv ID, and normalized title to prevent duplicate ingestion.
#
# Inputs / Outputs:
#   Inputs: NOTION_LIT_DB_ID, Notion API client
#   Outputs: existing_papers (list), dedup_indices (dict)
#
# Notes:
#   - Fetches all papers with pagination support
#   - Creates multiple deduplication indices for robust matching
#   - Title normalization: lowercase, strip whitespace, remove special chars
#   - Handles missing identifiers gracefully
#   - Large paper databases may require batching or caching
#

# --- Helper function for title normalization ---
def normalize_title(title: str) -> str:
    """Normalize paper title for deduplication matching."""
    if not title:
        return ""
    # Lowercase, strip, remove special characters, collapse whitespace
    normalized = title.lower().strip()
    normalized = re.sub(r'[^a-z0-9\s]', '', normalized)
    normalized = re.sub(r'\s+', ' ', normalized)
    return normalized

# --- Fetch all existing papers from Notion ---
log_info("Fetching existing papers from Notion for deduplication...")

existing_papers = []
has_more = True
start_cursor = None
page_count = 0

try:
    while has_more:
        query_params = {
            'database_id': NOTION_LIT_DB_ID,
            'page_size': 100
        }
        if start_cursor:
            query_params['start_cursor'] = start_cursor
        
        # --- Fetch all existing papers from Notion ---
        log_info("Fetching existing papers from Notion for deduplication...")
        
        existing_papers = []
        has_more = True
        start_cursor = None
        page_count = 0
        
        notion_version = os.getenv("NOTION_VERSION", "2022-06-28")
        headers = {
            "Authorization": f"Bearer {notion_token}",
            "Notion-Version": notion_version,
            "Content-Type": "application/json",
        }
        
        try:
            while has_more:
                payload = {"page_size": 100}
                if start_cursor:
                    payload["start_cursor"] = start_cursor
        
                url = f"https://api.notion.com/v1/databases/{NOTION_LIT_DB_ID}/query"
                r = requests.post(url, headers=headers, json=payload, timeout=30)
                if not r.ok:
                    log_error(f"Notion papers query failed: {r.status_code} {r.text}")
                    r.raise_for_status()
        
                response = r.json()
                results = response.get("results", [])
                existing_papers.extend(results)
        
                has_more = response.get("has_more", False)
                start_cursor = response.get("next_cursor")
                page_count += 1
        
                log_info(f"Fetched page {page_count} ({len(results)} papers)")
        
                if has_more:
                    time.sleep(0.3)
        
            log_info(f"Total existing papers fetched: {len(existing_papers)}")
        
        except Exception as e:
            log_error(f"Error fetching existing papers: {str(e)}")
            raise

        existing_papers.extend(response['results'])
        
        has_more = response.get('has_more', False)
        start_cursor = response.get('next_cursor')
        page_count += 1
        
        log_info(f"Fetched page {page_count} ({len(response['results'])} papers)")
        
        # Rate limiting: brief pause between pages
        if has_more:
            time.sleep(0.3)
    
    log_info(f"Total existing papers fetched: {len(existing_papers)}")

except Exception as e:
    log_error(f"Error fetching existing papers: {str(e)}")
    raise

# --- Build deduplication indices ---
log_info("Building deduplication indices...")

dedup_indices = {
    'doi': {},           # DOI -> page_id
    'openalex_id': {},   # OpenAlex ID -> page_id
    'arxiv_id': {},      # arXiv ID -> page_id
    'title': {}          # Normalized title -> page_id
}

for paper in existing_papers:
    page_id = paper['id']
    props = paper['properties']
    
    # Extract DOI
    doi_prop = props.get('DOI', {}).get('rich_text', [])
    if doi_prop:
        doi = doi_prop[0].get('plain_text', '').strip()
        if doi:
            dedup_indices['doi'][doi.lower()] = page_id
    
    # Extract OpenAlex ID
    openalex_prop = props.get('OpenAlex ID', {}).get('rich_text', [])
    if openalex_prop:
        openalex_id = openalex_prop[0].get('plain_text', '').strip()
        if openalex_id:
            dedup_indices['openalex_id'][openalex_id.lower()] = page_id
    
    # Extract arXiv ID
    arxiv_prop = props.get('arXiv ID', {}).get('rich_text', [])
    if arxiv_prop:
        arxiv_id = arxiv_prop[0].get('plain_text', '').strip()
        if arxiv_id:
            dedup_indices['arxiv_id'][arxiv_id.lower()] = page_id
    
    # Extract and normalize title
    title_prop = props.get('Title', {}).get('title', [])
    if title_prop:
        title = title_prop[0].get('plain_text', '').strip()
        if title:
            normalized = normalize_title(title)
            if normalized:
                dedup_indices['title'][normalized] = page_id

# --- Log index statistics ---
log_info(f"Deduplication indices built:")
log_info(f"  - DOI index: {len(dedup_indices['doi'])} entries")
log_info(f"  - OpenAlex ID index: {len(dedup_indices['openalex_id'])} entries")
log_info(f"  - arXiv ID index: {len(dedup_indices['arxiv_id'])} entries")
log_info(f"  - Title index: {len(dedup_indices['title'])} entries")

# --- Helper function for deduplication checking ---
def check_duplicate(paper_data: Dict[str, Any]) -> Optional[str]:
    """
    Check if a paper already exists in Notion.
    Returns existing page_id if duplicate found, None otherwise.
    
    Args:
        paper_data: Dict with keys: doi, openalex_id, arxiv_id, title
    
    Returns:
        page_id (str) if duplicate found, None otherwise
    """
    # Check DOI first (most reliable)
    doi = paper_data.get('doi', '').strip().lower()
    if doi and doi in dedup_indices['doi']:
        return dedup_indices['doi'][doi]
    
    # Check OpenAlex ID
    openalex_id = paper_data.get('openalex_id', '').strip().lower()
    if openalex_id and openalex_id in dedup_indices['openalex_id']:
        return dedup_indices['openalex_id'][openalex_id]
    
    # Check arXiv ID
    arxiv_id = paper_data.get('arxiv_id', '').strip().lower()
    if arxiv_id and arxiv_id in dedup_indices['arxiv_id']:
        return dedup_indices['arxiv_id'][arxiv_id]
    
    # Check normalized title (fallback)
    title = paper_data.get('title', '').strip()
    if title:
        normalized = normalize_title(title)
        if normalized and normalized in dedup_indices['title']:
            return dedup_indices['title'][normalized]
    
    return None

log_info("Deduplication check function ready")
log_info("Existing papers loaded successfully")


[INFO 2026-01-19T05:15:13.933818+00:00] Fetching existing papers from Notion for deduplication...
[INFO 2026-01-19T05:15:13.934948+00:00] Fetching existing papers from Notion for deduplication...
[INFO 2026-01-19T05:15:14.431093+00:00] Fetched page 1 (56 papers)
[INFO 2026-01-19T05:15:14.431273+00:00] Total existing papers fetched: 56
[INFO 2026-01-19T05:15:14.431322+00:00] Fetched page 2 (56 papers)
[INFO 2026-01-19T05:15:14.431338+00:00] Total existing papers fetched: 112
[INFO 2026-01-19T05:15:14.431457+00:00] Building deduplication indices...
[INFO 2026-01-19T05:15:14.431971+00:00] Deduplication indices built:
[INFO 2026-01-19T05:15:14.432049+00:00]   - DOI index: 0 entries
[INFO 2026-01-19T05:15:14.432108+00:00]   - OpenAlex ID index: 0 entries
[INFO 2026-01-19T05:15:14.432162+00:00]   - arXiv ID index: 0 entries
[INFO 2026-01-19T05:15:14.432215+00:00]   - Title index: 0 entries
[INFO 2026-01-19T05:15:14.432549+00:00] Deduplication check function ready
[INFO 2026-01-19T05:15:14.43

In [30]:
# ============================================================
# Cell 04 — Fetch existing Research Questions, Clusters, and Gaps
# ============================================================
# Overview:
#   Queries the Notion Research Questions, Clusters, and Gaps databases
#   to retrieve all existing entities for LLM-based paper classification.
#   Builds structured lookup indices and prepares context for relevance
#   matching in Cell 09.
#
# Inputs / Outputs:
#   Inputs: NOTION_RQ_DB_ID, NOTION_CLUSTERS_DB_ID, NOTION_GAPS_DB_ID
#   Outputs: research_questions (list), clusters (list), gaps (list),
#            classification_context (dict)
#
# Notes:
#   - Fetches all records with pagination support
#   - Extracts key fields: Title, Description, Status, Relations
#   - Filters out archived/completed items unless explicitly needed
#   - Prepares structured context for LLM prompts
#   - Handles missing or incomplete records gracefully
#
# --- Notion REST helpers (DB query + schema) ---
notion_version = os.getenv("NOTION_VERSION", "2022-06-28")
notion_headers = {
    "Authorization": f"Bearer {notion_token}",
    "Notion-Version": notion_version,
    "Content-Type": "application/json",
}

def notion_get_db(db_id: str) -> Dict[str, Any]:
    url = f"https://api.notion.com/v1/databases/{db_id}"
    r = requests.get(url, headers=notion_headers, timeout=30)
    if not r.ok:
        log_error(f"Notion db retrieve failed: {r.status_code} {r.text}")
        r.raise_for_status()
    return r.json()

def notion_query_all(db_id: str, payload: Optional[Dict[str, Any]] = None, page_size: int = 100) -> List[Dict[str, Any]]:
    """Fetch all pages from a Notion database with pagination (REST)."""
    results: List[Dict[str, Any]] = []
    has_more = True
    start_cursor = None
    page_count = 0

    base_payload = payload.copy() if payload else {}
    base_payload["page_size"] = page_size

    while has_more:
        p = dict(base_payload)
        if start_cursor:
            p["start_cursor"] = start_cursor

        url = f"https://api.notion.com/v1/databases/{db_id}/query"
        r = requests.post(url, headers=notion_headers, json=p, timeout=30)
        if not r.ok:
            log_error(f"Notion query failed (db={db_id}): {r.status_code} {r.text}")
            r.raise_for_status()

        data = r.json()
        batch = data.get("results", [])
        results.extend(batch)

        has_more = data.get("has_more", False)
        start_cursor = data.get("next_cursor")
        page_count += 1

        log_info(f"Fetched page {page_count} ({len(batch)} records) from db={db_id}")

        if has_more:
            time.sleep(0.3)

    return results

def detect_title_property_name(db_id: str) -> str:
    """Detect the Notion 'title' property name for a database (robust to Name/Title/日本語)."""
    db = notion_get_db(db_id)
    for prop_name, prop_def in db.get("properties", {}).items():
        if prop_def.get("type") == "title":
            return prop_name
    raise ValueError(f"Could not find a title property in database: {db_id}")

# --- Fetch Research Questions from Notion ---
# --- Fetch Research Questions / Clusters / Gaps (REST) ---
log_info("Fetching Research Questions from Notion...")
research_questions = notion_query_all(NOTION_RQ_DB_ID)
log_info(f"Total Research Questions fetched: {len(research_questions)}")

log_info("Fetching Clusters from Notion...")
clusters = notion_query_all(NOTION_CLUSTERS_DB_ID)
log_info(f"Total Clusters fetched: {len(clusters)}")

log_info("Fetching Gaps from Notion...")
gaps = notion_query_all(NOTION_GAPS_DB_ID)
log_info(f"Total Gaps fetched: {len(gaps)}")


# --- Extract structured data for classification ---
log_info("Extracting structured data for LLM classification...")

def extract_title(page: Dict) -> str:
    """Extract title from Notion page (robust: Title/Name/any title property)."""
    props = page.get('properties', {})

    # Common property names
    for key in ['Title', 'Name']:
        p = props.get(key, {})
        if p.get('type') == 'title':
            t = p.get('title') or []
            if t:
                return (t[0].get('plain_text') or '').strip()

    # Fallback: find any property whose type is 'title'
    for _, p in props.items():
        if isinstance(p, dict) and p.get('type') == 'title':
            t = p.get('title') or []
            if t:
                return (t[0].get('plain_text') or '').strip()

    return ""


def extract_description(page: Dict) -> str:
    """Extract description/rich text from Notion page."""
    for prop_name in ['Description', 'Summary', 'Details', 'Notes']:
        desc_prop = page['properties'].get(prop_name, {}).get('rich_text', [])
        if desc_prop:
            return desc_prop[0].get('plain_text', '').strip()
    return ""

def extract_status(page: Dict) -> str:
    """Extract status from Notion page."""
    status_prop = page['properties'].get('Status', {})
    if status_prop.get('select'):
        return status_prop['select'].get('name', '').strip()
    elif status_prop.get('status'):
        return status_prop['status'].get('name', '').strip()
    return ""

# --- Build structured lists for classification ---
rq_list = []
for rq in research_questions:
    title = extract_title(rq)
    description = extract_description(rq)
    status = extract_status(rq)
    
    if status.lower() in ['archived', 'completed', 'closed']:
        continue
    
    if title:
        rq_list.append({
            'page_id': rq['id'],
            'title': title,
            'description': description,
            'status': status,
            'type': 'research_question'
        })

cluster_list = []
for cluster in clusters:
    title = extract_title(cluster)
    description = extract_description(cluster)
    status = extract_status(cluster)
    
    if status.lower() in ['archived', 'completed', 'closed']:
        continue
    
    if title:
        cluster_list.append({
            'page_id': cluster['id'],
            'title': title,
            'description': description,
            'status': status,
            'type': 'cluster'
        })

gap_list = []
for gap in gaps:
    title = extract_title(gap)
    description = extract_description(gap)
    status = extract_status(gap)
    
    if status.lower() in ['archived', 'completed', 'closed']:
        continue
    
    if title:
        gap_list.append({
            'page_id': gap['id'],
            'title': title,
            'description': description,
            'status': status,
            'type': 'gap'
        })

# --- Build classification context for LLM ---
classification_context = {
    'research_questions': rq_list,
    'clusters': cluster_list,
    'gaps': gap_list,
    'total_active_rqs': len(rq_list),
    'total_active_clusters': len(cluster_list),
    'total_active_gaps': len(gap_list)
}

# --- Create lookup indices by page_id ---
rq_by_id = {rq['page_id']: rq for rq in rq_list}
cluster_by_id = {c['page_id']: c for c in cluster_list}
gap_by_id = {g['page_id']: g for g in gap_list}

# --- Log summary statistics ---
log_info(f"Classification context prepared:")
log_info(f"  - Active Research Questions: {len(rq_list)} / {len(research_questions)} total")
log_info(f"  - Active Clusters: {len(cluster_list)} / {len(clusters)} total")
log_info(f"  - Active Gaps: {len(gap_list)} / {len(gaps)} total")

if not rq_list:
    log_warning("No active Research Questions found!")
    log_warning("Paper classification will have no targets. Consider creating RQs first.")

log_info("Research Questions, Clusters, and Gaps loaded successfully")


[INFO 2026-01-19T05:51:52.611302+00:00] Fetching Research Questions from Notion...
[INFO 2026-01-19T05:51:53.764157+00:00] Fetched page 1 (30 records) from db=2a98e0e4d16280b7bad2cf6635a3ef17
[INFO 2026-01-19T05:51:53.768261+00:00] Total Research Questions fetched: 30
[INFO 2026-01-19T05:51:53.768380+00:00] Fetching Clusters from Notion...
[INFO 2026-01-19T05:51:53.962785+00:00] Fetched page 1 (0 records) from db=2ed8e0e4d16280cb9279cdbfa23e2a64
[INFO 2026-01-19T05:51:53.963313+00:00] Total Clusters fetched: 0
[INFO 2026-01-19T05:51:53.963426+00:00] Fetching Gaps from Notion...
[INFO 2026-01-19T05:51:54.381901+00:00] Fetched page 1 (32 records) from db=2ec8e0e4d162803f9044ce1fc508585d
[INFO 2026-01-19T05:51:54.385596+00:00] Total Gaps fetched: 32
[INFO 2026-01-19T05:51:54.385732+00:00] Extracting structured data for LLM classification...
[INFO 2026-01-19T05:51:54.387825+00:00] Classification context prepared:
[INFO 2026-01-19T05:51:54.387960+00:00]   - Active Research Questions: 30 / 3

In [24]:
# ============================================================
# Cell 05 — Scan OpenAlex for new papers since last run (themes + search)
# ============================================================

log_info("Scanning OpenAlex for new papers since last run...")

# -----------------------------
# Themes / keyword sets
# -----------------------------
THEMES: Dict[str, List[str]] = {
    "venture_capital": [
        "venture capital", "VC", "venture fund", "venture investing",
        "startup financing", "early-stage financing"
    ],
    "limited_partners": [
        "limited partner", "LP", "institutional investor",
        "fund of funds", "pension fund", "endowment"
    ],
    "government_vc": [
        "government venture capital", "public venture capital",
        "state-backed venture capital", "sovereign wealth fund",
        "innovation agency", "development finance institution"
    ],
    "entrepreneurship_policy": [
        "entrepreneurship policy", "innovation policy",
        "startup policy", "industrial policy",
        "regulation", "tax incentive", "public subsidy"
    ],
}

def build_openalex_search_query(themes: Dict[str, List[str]]) -> str:
    """
    Safer OpenAlex search query:
    - Removes overly broad acronyms (VC, LP)
    - Removes overly broad singletons (regulation)
    - Prefers phrase queries for multi-word concepts
    """
    DROP = {"vc", "lp", "regulation"}  # too broad / ambiguous
    KEEP_SINGLE = {"endowment"}        # single word but ok-ish

    terms = []
    for _, kws in themes.items():
        for kw in kws:
            kw = (kw or "").strip()
            if not kw:
                continue
            if kw.lower() in DROP:
                continue
            if " " in kw:
                terms.append(f'"{kw}"')  # phrase
            else:
                if kw.lower() in KEEP_SINGLE:
                    terms.append(kw)
                else:
                    # single words are often too broad; skip by default
                    continue

    # de-dup
    seen = set()
    deduped = []
    for t in terms:
        tl = t.lower()
        if tl not in seen:
            seen.add(tl)
            deduped.append(t)

    # IMPORTANT: fewer terms = fewer false positives
    deduped = deduped[:20]

    return " OR ".join(deduped)


# --- Configuration ---
OPENALEX_BASE_URL = "https://api.openalex.org/works"
RESULTS_PER_PAGE = 200
MAX_PAGES = 10
RATE_LIMIT_DELAY = 0.11

# --- Date window ---
from_date_str = last_run_timestamp.strftime('%Y-%m-%d')
to_date_str = datetime.now(timezone.utc).strftime('%Y-%m-%d')
log_info(f"Filtering papers from publication date: {from_date_str} to {to_date_str}")

SEARCH_Q = build_openalex_search_query(THEMES)
log_info("Theme search enabled (search=... OR ...).")

base_params = {
    'mailto': openalex_email,
    # IMPORTANT: use `search` for OR across terms (title/abstract/etc)
    'search': SEARCH_Q,
    # keep filters for date only
    'filter': f'from_publication_date:{from_date_str},to_publication_date:{to_date_str}',
    'sort': 'publication_date:desc',
    'per_page': RESULTS_PER_PAGE,
    'page': 1
}

openalex_papers = []
total_fetched = 0
page_num = 1
hit_max_pages = False

try:
    while page_num <= MAX_PAGES:
        log_info(f"Fetching OpenAlex page {page_num}...")

        params = base_params.copy()
        params['page'] = page_num

        response = requests.get(
            OPENALEX_BASE_URL,
            params=params,
            headers={'User-Agent': f'DailyScannerBot/1.0 (mailto:{openalex_email})'},
            timeout=30
        )

        if page_num == 1:
            log_info(f"OpenAlex request URL: {response.url}")

        if response.status_code != 200:
            log_error(f"OpenAlex API error: HTTP {response.status_code}")
            log_error(f"Response: {response.text[:800]}")
            break

        data = response.json()
        results = data.get('results') or []

        if not results:
            log_info("No more results from OpenAlex")
            break

        extracted_this_page = 0
        for work in results:
            try:
                if not isinstance(work, dict):
                    continue

                openalex_id = (work.get('id') or '').replace('https://openalex.org/', '')
                doi_raw = work.get('doi') or ''
                doi = doi_raw.replace('https://doi.org/', '') if doi_raw else ''

                title = (work.get('title') or '').strip()
                if not title:
                    continue

                authors_list = []
                for authorship in (work.get('authorships') or []):
                    if not isinstance(authorship, dict):
                        continue
                    author = authorship.get('author') or {}
                    if not isinstance(author, dict):
                        continue
                    name = author.get('display_name')
                    if name:
                        authors_list.append(name)
                authors = ', '.join(authors_list) if authors_list else 'Unknown'

                abstract = ''
                abstract_inverted = work.get('abstract_inverted_index')
                if isinstance(abstract_inverted, dict) and abstract_inverted:
                    word_positions = []
                    for word, positions in abstract_inverted.items():
                        if not isinstance(positions, list):
                            continue
                        for pos in positions:
                            if isinstance(pos, int):
                                word_positions.append((pos, word))
                    word_positions.sort()
                    abstract = ' '.join([word for _, word in word_positions])
                else:
                    abstract = work.get('abstract') or ''

                pub_date_str = work.get('publication_date') or ''
                pub_date = None
                if pub_date_str:
                    try:
                        pub_date = datetime.fromisoformat(pub_date_str)
                        if pub_date.tzinfo is None:
                            pub_date = pub_date.replace(tzinfo=timezone.utc)
                    except Exception:
                        pub_date = None

                venue = ''
                primary_location = work.get('primary_location') or {}
                if isinstance(primary_location, dict):
                    source = primary_location.get('source') or {}
                    if isinstance(source, dict):
                        venue = source.get('display_name') or ''

                url = doi_raw if doi_raw else (f"https://openalex.org/{openalex_id}" if openalex_id else '')

                openalex_papers.append({
                    'source': 'OpenAlex',
                    'openalex_id': openalex_id,
                    'doi': doi,
                    'arxiv_id': '',
                    'title': title,
                    'authors': authors,
                    'abstract': abstract,
                    'publication_date': pub_date,
                    'venue': venue,
                    'url': url,
                    'raw_metadata': work
                })
                extracted_this_page += 1

            except Exception as e:
                log_error(f"Error parsing OpenAlex work: {str(e)}")
                continue

        total_fetched += len(results)
        log_info(
            f"Extracted {extracted_this_page} papers from page {page_num} "
            f"(page size: {len(results)}, total fetched: {total_fetched})"
        )

        meta = data.get('meta') or {}
        total_results = meta.get('count', 0) or 0
        per_page = meta.get('per_page', RESULTS_PER_PAGE) or RESULTS_PER_PAGE

        if total_fetched >= total_results or len(results) < per_page:
            log_info("All available results fetched")
            break

        page_num += 1
        if page_num <= MAX_PAGES:
            time.sleep(RATE_LIMIT_DELAY)

    if page_num == MAX_PAGES:
        hit_max_pages = True

    log_info(f"OpenAlex scan complete: {len(openalex_papers)} papers retrieved")

    if hit_max_pages:
        log_warning(f"Reached MAX_PAGES limit ({MAX_PAGES}). Some results may be missing.")
        log_warning("Consider increasing MAX_PAGES or narrowing date range / adding additional filters.")

except requests.exceptions.RequestException as e:
    log_error(f"Network error querying OpenAlex: {str(e)}")
    log_error("Continuing with empty OpenAlex results")
    openalex_papers = []

except Exception as e:
    log_error(f"Unexpected error during OpenAlex scan: {str(e)}")
    log_error("Continuing with empty OpenAlex results")
    openalex_papers = []

# --- Summary statistics ---
if openalex_papers:
    with_abstracts = sum(1 for p in openalex_papers if p.get('abstract'))
    with_dois = sum(1 for p in openalex_papers if p.get('doi'))

    log_info(f"OpenAlex papers retrieved: {len(openalex_papers)}")
    log_info(f"  - With abstracts: {with_abstracts} ({100*with_abstracts//len(openalex_papers)}%)")
    log_info(f"  - With DOIs: {with_dois} ({100*with_dois//len(openalex_papers)}%)")

    dates = [p['publication_date'] for p in openalex_papers if p.get('publication_date')]
    if dates:
        earliest = min(dates)
        latest = max(dates)
        log_info(f"  - Date range: {earliest.date()} to {latest.date()}")
else:
    log_info("No papers retrieved from OpenAlex (this may be expected for recent runs)")

log_info("OpenAlex scan complete")


[INFO 2026-01-19T05:37:45.776002+00:00] Scanning OpenAlex for new papers since last run...
[INFO 2026-01-19T05:37:45.777235+00:00] Filtering papers from publication date: 2026-01-12 to 2026-01-19
[INFO 2026-01-19T05:37:45.777392+00:00] Theme search enabled (search=... OR ...).
[INFO 2026-01-19T05:37:45.909720+00:00] Fetching OpenAlex page 1...
[INFO 2026-01-19T05:37:48.169636+00:00] OpenAlex request URL: https://api.openalex.org/works?mailto=user%40example.com&search=%22venture+capital%22+OR+%22venture+fund%22+OR+%22venture+investing%22+OR+%22startup+financing%22+OR+%22early-stage+financing%22+OR+%22limited+partner%22+OR+%22institutional+investor%22+OR+%22fund+of+funds%22+OR+%22pension+fund%22+OR+endowment+OR+%22government+venture+capital%22+OR+%22public+venture+capital%22+OR+%22state-backed+venture+capital%22+OR+%22sovereign+wealth+fund%22+OR+%22innovation+agency%22+OR+%22development+finance+institution%22+OR+%22entrepreneurship+policy%22+OR+%22innovation+policy%22+OR+%22startup+polic

In [25]:
# ============================================================
# Cell 06 — Scan arXiv for new papers since last run (themes + patched)
# ============================================================

log_info("Scanning arXiv for new papers since last run...")

# -----------------------------
# Themes / keyword sets
# -----------------------------
# Keep these relatively broad for recall; downstream steps can filter harder.
THEMES: Dict[str, List[str]] = {
    "venture_capital": [
        "venture capital", "VC", "venture fund", "venture investing",
        "startup financing", "early-stage financing"
    ],
    "limited_partners": [
        "limited partner", "LP", "institutional investor",
        "fund of funds", "pension fund", "endowment"
    ],
    "government_vc": [
        "government venture capital", "public venture capital",
        "state-backed venture capital", "sovereign wealth fund",
        "innovation agency", "development finance institution"
    ],
    "entrepreneurship_policy": [
        "entrepreneurship policy", "innovation policy",
        "startup policy", "industrial policy",
        "regulation", "tax incentive", "public subsidy"
    ],
}

def build_arxiv_theme_query(themes: Dict[str, List[str]], max_terms: int = 20) -> str:
    """
    Convert THEMES dict into arXiv-compatible keyword query.
    - Uses all:"..." which matches across fields (title/abstract/etc).
    - Drops overly broad / ambiguous acronyms by default (VC, LP) and generic terms (regulation).
    - Caps total number of terms to keep the query string manageable.
    """
    DROP = {"vc", "lp", "regulation"}  # too broad / ambiguous
    KEEP_SINGLE = {"endowment"}        # single word but acceptable

    terms: List[str] = []
    for _, kws in themes.items():
        for kw in kws:
            kw = (kw or "").strip()
            if not kw:
                continue
            if kw.lower() in DROP:
                continue

            # Prefer phrase queries; single-words are often noisy
            if " " in kw:
                terms.append(f'all:"{kw}"')
            else:
                if kw.lower() in KEEP_SINGLE:
                    terms.append(f'all:"{kw}"')
                else:
                    continue

    # de-dup preserve order
    seen = set()
    deduped = []
    for t in terms:
        tl = t.lower()
        if tl not in seen:
            seen.add(tl)
            deduped.append(t)

    deduped = deduped[:max_terms]
    return " OR ".join(deduped)

# --- Configuration ---
ARXIV_BASE_URL = "http://export.arxiv.org/api/query"
RESULTS_PER_PAGE = 100   # arXiv recommends <= 100-200 per request
MAX_RESULTS = 300        # Safety limit for daily ops (tune as needed)
RATE_LIMIT_DELAY = 3.0   # 3 seconds between requests (arXiv guideline)

# --- Date window ---
from_date_arxiv = last_run_timestamp.strftime('%Y%m%d%H%M%S')
to_date_arxiv = config_state['current_timestamp'].strftime('%Y%m%d%H%M%S')
log_info(f"Filtering papers updated from: {from_date_arxiv} to {to_date_arxiv}")

# --- Theme query + date filter ---
THEME_QUERY = build_arxiv_theme_query(THEMES)
if not THEME_QUERY:
    log_warning("Theme query is empty; falling back to date-only scan (may be large).")
    search_query = f"lastUpdatedDate:[{from_date_arxiv} TO {to_date_arxiv}]"
else:
    # Using lastUpdatedDate keeps revisions; switch to submittedDate if you want only new submissions.
    # submittedDate version:
    # search_query = f"({THEME_QUERY}) AND submittedDate:[{from_date_arxiv} TO {to_date_arxiv}]"
    search_query = f"({THEME_QUERY}) AND lastUpdatedDate:[{from_date_arxiv} TO {to_date_arxiv}]"

log_info("Theme search enabled for arXiv.")

# --- Initialize results container ---
arxiv_papers = []
total_fetched = 0
start_index = 0
hit_max_results = False

# --- XML namespace for parsing Atom feed ---
namespaces = {
    'atom': 'http://www.w3.org/2005/Atom',
    'arxiv': 'http://arxiv.org/schemas/atom'
}

try:
    while total_fetched < MAX_RESULTS:
        log_info(f"Fetching arXiv results starting at index {start_index}...")

        params = {
            'search_query': search_query,
            'start': start_index,
            'max_results': RESULTS_PER_PAGE,
            'sortBy': 'lastUpdatedDate',
            'sortOrder': 'descending'
        }

        query_url = f"{ARXIV_BASE_URL}?{urllib.parse.urlencode(params)}"

        # Debug helper: show URL once
        if start_index == 0:
            log_info(f"arXiv request URL: {query_url[:300]}...")  # avoid overly long log

        try:
            with urllib.request.urlopen(query_url, timeout=30) as response:
                xml_data = response.read()
        except Exception as e:
            log_error(f"arXiv API request error: {str(e)}")
            break

        try:
            root = ET.fromstring(xml_data)
        except ET.ParseError as e:
            log_error(f"XML parsing error: {str(e)}")
            break

        entries = root.findall('atom:entry', namespaces)

        if not entries:
            log_info("No more results from arXiv")
            break

        extracted_this_batch = 0

        for entry in entries:
            try:
                if entry is None:
                    continue

                entry_id = entry.find('atom:id', namespaces)
                arxiv_url = (entry_id.text or '').strip() if entry_id is not None else ''
                arxiv_id = arxiv_url.replace('http://arxiv.org/abs/', '').strip()
                if not arxiv_id:
                    continue

                title_elem = entry.find('atom:title', namespaces)
                title = (title_elem.text or '').strip().replace('\n', ' ') if title_elem is not None else ''
                if not title:
                    continue

                authors_list = []
                for author in entry.findall('atom:author', namespaces):
                    name_elem = author.find('atom:name', namespaces)
                    if name_elem is not None and name_elem.text:
                        authors_list.append(name_elem.text.strip())
                authors = ', '.join(authors_list) if authors_list else 'Unknown'

                summary_elem = entry.find('atom:summary', namespaces)
                abstract = (summary_elem.text or '').strip().replace('\n', ' ') if summary_elem is not None else ''

                doi = ''
                doi_elem = entry.find('arxiv:doi', namespaces)
                if doi_elem is not None and doi_elem.text:
                    doi = doi_elem.text.strip()

                published_elem = entry.find('atom:published', namespaces)
                updated_elem = entry.find('atom:updated', namespaces)

                pub_date = None
                if updated_elem is not None and updated_elem.text:
                    try:
                        pub_date = datetime.fromisoformat(updated_elem.text.replace('Z', '+00:00'))
                    except Exception:
                        pub_date = None

                if pub_date is None and published_elem is not None and published_elem.text:
                    try:
                        pub_date = datetime.fromisoformat(published_elem.text.replace('Z', '+00:00'))
                    except Exception:
                        pub_date = None

                categories = []
                for cat in entry.findall('atom:category', namespaces):
                    term = cat.get('term')
                    if term:
                        categories.append(term)

                venue = ', '.join(categories) if categories else 'arXiv'
                pdf_url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"

                arxiv_papers.append({
                    'source': 'arXiv',
                    'openalex_id': '',
                    'doi': doi,
                    'arxiv_id': arxiv_id,
                    'title': title,
                    'authors': authors,
                    'abstract': abstract,
                    'publication_date': pub_date,
                    'venue': venue,
                    'url': arxiv_url,
                    'pdf_url': pdf_url,
                    'categories': categories
                })
                extracted_this_batch += 1

            except Exception as e:
                log_error(f"Error parsing arXiv entry: {str(e)}")
                continue

        total_fetched += len(entries)
        log_info(f"Extracted {extracted_this_batch} papers from this batch (entries: {len(entries)}, total: {total_fetched})")

        # If fewer results than requested, we're done
        if len(entries) < RESULTS_PER_PAGE:
            log_info("Fewer results than requested; all available results fetched")
            break

        # Next page
        start_index += RESULTS_PER_PAGE

        # Rate limiting
        if total_fetched < MAX_RESULTS:
            log_info(f"Rate limiting: waiting {RATE_LIMIT_DELAY} seconds...")
            time.sleep(RATE_LIMIT_DELAY)

    if total_fetched >= MAX_RESULTS:
        hit_max_results = True

    log_info(f"arXiv scan complete: {len(arxiv_papers)} papers retrieved")

    if hit_max_results:
        log_warning(f"Reached MAX_RESULTS limit ({MAX_RESULTS}). Some results may be missing.")
        log_warning("Consider increasing MAX_RESULTS or narrowing the theme keywords / date range.")
        log_warning("If you only want new submissions (not revisions), switch lastUpdatedDate -> submittedDate.")

except Exception as e:
    log_error(f"Unexpected error during arXiv scan: {str(e)}")
    log_error("Continuing with empty arXiv results")
    arxiv_papers = []

# --- Summary statistics ---
if arxiv_papers:
    with_abstracts = sum(1 for p in arxiv_papers if p.get('abstract'))
    with_dois = sum(1 for p in arxiv_papers if p.get('doi'))

    log_info(f"arXiv papers retrieved: {len(arxiv_papers)}")
    log_info(f"  - With abstracts: {with_abstracts} ({100*with_abstracts//len(arxiv_papers)}%)")
    log_info(f"  - With DOIs: {with_dois} ({100*with_dois//len(arxiv_papers)}%)")

    dates = [p['publication_date'] for p in arxiv_papers if p.get('publication_date')]
    if dates:
        earliest = min(dates)
        latest = max(dates)
        log_info(f"  - Date range: {earliest.date()} to {latest.date()}")

    all_categories = []
    for p in arxiv_papers:
        all_categories.extend(p.get('categories', []))
    if all_categories:
        top_categories = Counter(all_categories).most_common(5)
        log_info(f"  - Top categories: {', '.join([f'{cat}({count})' for cat, count in top_categories])}")
else:
    log_info("No papers retrieved from arXiv (this may be expected for recent runs)")

log_info("arXiv scan complete")


[INFO 2026-01-19T05:39:43.544698+00:00] Scanning arXiv for new papers since last run...
[INFO 2026-01-19T05:39:43.546434+00:00] Filtering papers updated from: 20260112051309 to 20260119051309
[INFO 2026-01-19T05:39:43.546643+00:00] Theme search enabled for arXiv.
[INFO 2026-01-19T05:39:43.556576+00:00] Fetching arXiv results starting at index 0...
[INFO 2026-01-19T05:39:43.556973+00:00] arXiv request URL: http://export.arxiv.org/api/query?search_query=%28all%3A%22venture+capital%22+OR+all%3A%22venture+fund%22+OR+all%3A%22venture+investing%22+OR+all%3A%22startup+financing%22+OR+all%3A%22early-stage+financing%22+OR+all%3A%22limited+partner%22+OR+all%3A%22institutional+investor%22+OR+all%3A%22fund+of+fu...
[INFO 2026-01-19T05:39:45.320185+00:00] Extracted 21 papers from this batch (entries: 21, total: 21)
[INFO 2026-01-19T05:39:45.320383+00:00] Fewer results than requested; all available results fetched
[INFO 2026-01-19T05:39:45.320414+00:00] arXiv scan complete: 21 papers retrieved
[INFO

In [27]:
# ============================================================
# Cell 07 — Optional Google Drive PDF ingestion (env-aligned + debug)
# ============================================================

log_info("Starting optional Google Drive PDF ingestion...")

gdrive_papers = []

# --- Read config from env (aligned) ---
DRIVE_FOLDER_ID = os.getenv("DRIVE_FOLDER_ID", "") or os.getenv("GDRIVE_FOLDER_ID", "")
GOOGLE_OAUTH_CLIENT_SECRET_JSON = os.getenv("GOOGLE_OAUTH_CLIENT_SECRET_JSON", "")
GOOGLE_TOKEN_JSON = os.getenv("GOOGLE_TOKEN_JSON", "")

GDRIVE_ENABLED = bool(DRIVE_FOLDER_ID)
if not GDRIVE_ENABLED:
    log_info("Google Drive ingestion disabled (DRIVE_FOLDER_ID not configured)")
    log_info("Skipping Google Drive scan")
else:
    log_info(f"Google Drive ingestion enabled for folder: {DRIVE_FOLDER_ID}")

    try:
        from googleapiclient.discovery import build
        from google.auth.transport.requests import Request
        from google.oauth2.credentials import Credentials
        from google_auth_oauthlib.flow import InstalledAppFlow
    except ImportError as e:
        log_error(f"Google Drive libraries not installed: {str(e)}")
        log_error("Install with: pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib")
        GDRIVE_ENABLED = False

    if GDRIVE_ENABLED:
        # --- Scopes ---
        SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]

        # --- Load credentials ---
        creds = None
        token_path = GOOGLE_TOKEN_JSON or "token.json"
        client_secret_path = GOOGLE_OAUTH_CLIENT_SECRET_JSON or "credentials.json"

        log_info(f"Token path: {token_path}")
        log_info(f"Client secret path: {client_secret_path}")
        log_info(f"Scopes: {SCOPES}")

        try:
            if os.path.exists(token_path):
                log_info(f"Loading OAuth token from: {token_path}")
                creds = Credentials.from_authorized_user_file(token_path, SCOPES)

            if not creds or not creds.valid:
                if creds and creds.expired and creds.refresh_token:
                    log_info("Refreshing expired credentials...")
                    try:
                        creds.refresh(Request())
                    except Exception as e:
                        # Typical failure: invalid_scope due to stale refresh_token
                        log_error(f"Credential refresh failed: {str(e)}")
                        log_warning("This often happens when token.json was created with different scopes.")
                        log_warning("Fix: delete token.json and re-run OAuth flow to generate a new token.")
                        creds = None

                if not creds:
                    if not os.path.exists(client_secret_path):
                        log_error("OAuth client secret JSON not found.")
                        log_error("Set GOOGLE_OAUTH_CLIENT_SECRET_JSON to the downloaded client_secret.json path.")
                        raise FileNotFoundError(client_secret_path)

                    log_info("Starting OAuth flow (requires browser interaction on first run)...")
                    flow = InstalledAppFlow.from_client_secrets_file(client_secret_path, SCOPES)
                    creds = flow.run_local_server(port=0)

                # Save token for next time
                if creds and creds.valid:
                    os.makedirs(os.path.dirname(token_path) or ".", exist_ok=True)
                    with open(token_path, "w") as token:
                        token.write(creds.to_json())
                    log_info(f"Saved OAuth token to: {token_path}")

            if not creds or not creds.valid:
                log_error("Failed to obtain valid Google Drive credentials; skipping Drive scan.")
                GDRIVE_ENABLED = False

        except Exception as e:
            log_error(f"Unexpected credential error: {str(e)}")
            log_warning("If you see invalid_scope, delete token.json and re-auth.")
            GDRIVE_ENABLED = False

    if GDRIVE_ENABLED:
        try:
            log_info("Building Google Drive API service...")
            service = build("drive", "v3", credentials=creds)

            from_date_rfc3339 = last_run_timestamp.strftime('%Y-%m-%dT%H:%M:%S.000Z')
            query = (
                f"'{DRIVE_FOLDER_ID}' in parents "
                f"and mimeType='application/pdf' "
                f"and modifiedTime > '{from_date_rfc3339}' "
                f"and trashed=false"
            )

            log_info(f"Querying for PDFs modified after: {from_date_rfc3339}")

            page_token = None
            while True:
                results = service.files().list(
                    q=query,
                    spaces="drive",
                    fields="nextPageToken, files(id, name, modifiedTime, size, webViewLink, owners)",
                    pageSize=100,
                    pageToken=page_token
                ).execute()

                files = results.get("files", [])
                if not files:
                    break

                log_info(f"Found {len(files)} PDF files in this batch")

                for f in files:
                    file_id = f.get("id", "")
                    file_name = f.get("name", "Unknown")
                    modified_time = f.get("modifiedTime", "")
                    file_size = f.get("size", 0)
                    web_link = f.get("webViewLink", "")

                    pub_date = None
                    if modified_time:
                        try:
                            pub_date = datetime.fromisoformat(modified_time.replace("Z", "+00:00"))
                        except Exception:
                            pub_date = None

                    owners = f.get("owners") or []
                    owner_names = []
                    for o in owners:
                        if isinstance(o, dict):
                            owner_names.append(o.get("displayName", "Unknown"))
                    authors = ", ".join(owner_names) if owner_names else "Unknown"

                    title = file_name[:-4] if file_name.lower().endswith(".pdf") else file_name

                    gdrive_papers.append({
                        "source": "Google Drive",
                        "openalex_id": "",
                        "doi": "",
                        "arxiv_id": "",
                        "title": title,
                        "authors": authors,
                        "abstract": "",
                        "publication_date": pub_date,
                        "venue": "Google Drive Upload",
                        "url": web_link,
                        "file_id": file_id,
                        "file_name": file_name,
                        "file_size": int(file_size) if file_size else 0,
                    })

                page_token = results.get("nextPageToken")
                if not page_token:
                    break

                time.sleep(0.1)

            log_info(f"Google Drive scan complete: {len(gdrive_papers)} PDFs found")

        except Exception as e:
            log_error(f"Unexpected error during Google Drive scan: {str(e)}")
            log_info("Continuing with empty Google Drive results")
            gdrive_papers = []

# --- Summary ---
if gdrive_papers:
    log_info(f"Google Drive papers retrieved: {len(gdrive_papers)}")
else:
    if GDRIVE_ENABLED:
        log_info("No new PDFs found in Google Drive folder")
    else:
        log_info("Google Drive scan skipped (disabled or credentials unavailable)")

log_info("Google Drive PDF ingestion complete (graceful failure mode)")


[INFO 2026-01-19T05:43:00.965057+00:00] Starting optional Google Drive PDF ingestion...
[INFO 2026-01-19T05:43:00.966966+00:00] Google Drive ingestion enabled for folder: 1SygzpVjCuk-_8oHk9XQOponn7T3ZOsgh
[INFO 2026-01-19T05:43:00.967079+00:00] Token path: token.json
[INFO 2026-01-19T05:43:00.967114+00:00] Client secret path: client_secret_750875982200-85rnsoqhr2af2b13peueev0bm60q22sh.apps.googleusercontent.com.json
[INFO 2026-01-19T05:43:00.967136+00:00] Scopes: ['https://www.googleapis.com/auth/drive.readonly']
[INFO 2026-01-19T05:43:00.967400+00:00] Loading OAuth token from: token.json
[INFO 2026-01-19T05:43:00.969255+00:00] Refreshing expired credentials...
[ERROR 2026-01-19T05:43:01.148555+00:00] Credential refresh failed: ('invalid_scope: Bad Request', {'error': 'invalid_scope', 'error_description': 'Bad Request'})
[WARN 2026-01-19T05:43:01.148685+00:00] This often happens when token.json was created with different scopes.
[WARN 2026-01-19T05:43:01.148709+00:00] Fix: delete token

In [28]:
# ============================================================
# Cell 08 — Deduplicate and merge new papers from all sources
# ============================================================
# Overview:
#   Combines papers from OpenAlex, arXiv, and Google Drive sources,
#   applies deduplication logic using the indices built in Cell 03,
#   and produces a unified list of new candidate papers for classification.
#   Handles conflicts when same paper appears in multiple sources by
#   preferring the most complete metadata.
#
# Inputs / Outputs:
#   Inputs: openalex_papers, arxiv_papers, gdrive_papers, dedup_indices
#   Outputs: new_papers (list), duplicate_count (int), merge_stats (dict)
#
# Notes:
#   - Deduplication priority: existing Notion papers > new papers
#   - Merges papers from multiple sources by DOI/OpenAlex/arXiv ID
#   - Prefers OpenAlex metadata > arXiv > Google Drive for conflicts
#   - Keeps track of all source origins for each paper
#   - Creates unified paper records with best available metadata
#   - Logs detailed statistics on deduplication and merging

log_info("Starting deduplication and merge of papers from all sources...")

# --- Combine all source papers into single list ---
all_source_papers = []
all_source_papers.extend(openalex_papers)
all_source_papers.extend(arxiv_papers)
all_source_papers.extend(gdrive_papers)

log_info(f"Total papers from all sources: {len(all_source_papers)}")
log_info(f"  - OpenAlex: {len(openalex_papers)}")
log_info(f"  - arXiv: {len(arxiv_papers)}")
log_info(f"  - Google Drive: {len(gdrive_papers)}")

# --- Check against existing Notion papers ---
log_info("Checking for duplicates against existing Notion papers...")

duplicate_count = 0
existing_duplicates = []

for paper in all_source_papers:
    existing_page_id = check_duplicate(paper)
    if existing_page_id:
        duplicate_count += 1
        existing_duplicates.append({
            'title': paper.get('title', 'Unknown'),
            'source': paper.get('source', 'Unknown'),
            'existing_page_id': existing_page_id
        })

log_info(f"Found {duplicate_count} papers already in Notion (will skip)")

# --- Filter out existing duplicates ---
new_candidate_papers = []
for paper in all_source_papers:
    if not check_duplicate(paper):
        new_candidate_papers.append(paper)

log_info(f"New candidate papers after Notion deduplication: {len(new_candidate_papers)}")

# --- Deduplicate within new candidates across sources ---
log_info("Deduplicating within new candidate papers...")

def get_paper_key(paper: Dict[str, Any]) -> Tuple[str, str, str, str]:
    """Extract deduplication keys from paper."""
    doi = paper.get('doi', '').strip().lower()
    openalex_id = paper.get('openalex_id', '').strip().lower()
    arxiv_id = paper.get('arxiv_id', '').strip().lower()
    title = normalize_title(paper.get('title', ''))
    return (doi, openalex_id, arxiv_id, title)

def merge_paper_metadata(papers: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Merge metadata from multiple source papers into single record.
    Priority: OpenAlex > arXiv > Google Drive for conflicts.
    """
    # Sort by source priority
    source_priority = {'OpenAlex': 0, 'arXiv': 1, 'Google Drive': 2}
    sorted_papers = sorted(papers, key=lambda p: source_priority.get(p.get('source', ''), 99))
    
    # Start with highest priority paper as base
    merged = sorted_papers[0].copy()
    
    # Collect all sources
    merged['sources'] = [p.get('source', 'Unknown') for p in papers]
    
    # Merge fields from other sources if missing in base
    for paper in sorted_papers[1:]:
        # Fill missing identifiers
        if not merged.get('doi') and paper.get('doi'):
            merged['doi'] = paper['doi']
        if not merged.get('openalex_id') and paper.get('openalex_id'):
            merged['openalex_id'] = paper['openalex_id']
        if not merged.get('arxiv_id') and paper.get('arxiv_id'):
            merged['arxiv_id'] = paper['arxiv_id']
        
        # Fill missing abstract (prefer longer/more complete)
        if not merged.get('abstract') and paper.get('abstract'):
            merged['abstract'] = paper['abstract']
        elif merged.get('abstract') and paper.get('abstract'):
            if len(paper['abstract']) > len(merged['abstract']):
                merged['abstract'] = paper['abstract']
        
        # Fill missing authors
        if not merged.get('authors') or merged.get('authors') == 'Unknown':
            if paper.get('authors') and paper.get('authors') != 'Unknown':
                merged['authors'] = paper['authors']
        
        # Fill missing publication date
        if not merged.get('publication_date') and paper.get('publication_date'):
            merged['publication_date'] = paper['publication_date']
        
        # Fill missing venue
        if not merged.get('venue') and paper.get('venue'):
            merged['venue'] = paper['venue']
        
        # Collect all URLs
        if 'all_urls' not in merged:
            merged['all_urls'] = []
        if merged.get('url') and merged['url'] not in merged['all_urls']:
            merged['all_urls'].append(merged['url'])
        if paper.get('url') and paper['url'] not in merged['all_urls']:
            merged['all_urls'].append(paper['url'])
        if paper.get('pdf_url') and paper['pdf_url'] not in merged.get('all_urls', []):
            merged.setdefault('all_urls', []).append(paper['pdf_url'])
    
    return merged

# --- Group papers by identity ---
paper_groups = defaultdict(list)
processed_indices = set()

for idx, paper in enumerate(new_candidate_papers):
    if idx in processed_indices:
        continue
    
    doi, openalex_id, arxiv_id, title = get_paper_key(paper)
    
    # Find matching papers
    matching_papers = [paper]
    processed_indices.add(idx)
    
    # Check remaining papers for matches
    for other_idx in range(idx + 1, len(new_candidate_papers)):
        if other_idx in processed_indices:
            continue
        
        other_paper = new_candidate_papers[other_idx]
        other_doi, other_openalex_id, other_arxiv_id, other_title = get_paper_key(other_paper)
        
        # Check for identity match
        is_match = False
        
        if doi and other_doi and doi == other_doi:
            is_match = True
        elif openalex_id and other_openalex_id and openalex_id == other_openalex_id:
            is_match = True
        elif arxiv_id and other_arxiv_id and arxiv_id == other_arxiv_id:
            is_match = True
        elif title and other_title and title == other_title:
            # Title match only if both have no other identifiers
            if not (doi or openalex_id or arxiv_id or other_doi or other_openalex_id or other_arxiv_id):
                is_match = True
        
        if is_match:
            matching_papers.append(other_paper)
            processed_indices.add(other_idx)
    
    # Create identity key for grouping
    identity_key = doi or openalex_id or arxiv_id or title or f"unknown_{idx}"
    paper_groups[identity_key] = matching_papers

log_info(f"Grouped {len(new_candidate_papers)} candidates into {len(paper_groups)} unique papers")

# --- Merge grouped papers ---
new_papers = []
cross_source_merges = 0

for identity_key, papers in paper_groups.items():
    if len(papers) > 1:
        cross_source_merges += 1
        sources = [p.get('source', 'Unknown') for p in papers]
        log_info(f"Merging {len(papers)} papers from sources: {', '.join(sources)}")
    
    merged_paper = merge_paper_metadata(papers)
    new_papers.append(merged_paper)

log_info(f"Cross-source merges performed: {cross_source_merges}")
log_info(f"Final unique new papers: {len(new_papers)}")

# --- Compile merge statistics ---
merge_stats = {
    'total_source_papers': len(all_source_papers),
    'openalex_papers': len(openalex_papers),
    'arxiv_papers': len(arxiv_papers),
    'gdrive_papers': len(gdrive_papers),
    'existing_duplicates': duplicate_count,
    'new_candidates': len(new_candidate_papers),
    'cross_source_merges': cross_source_merges,
    'final_new_papers': len(new_papers)
}

# --- Analyze new papers ---
if new_papers:
    with_abstracts = sum(1 for p in new_papers if p.get('abstract'))
    with_dois = sum(1 for p in new_papers if p.get('doi'))
    with_openalex = sum(1 for p in new_papers if p.get('openalex_id'))
    with_arxiv = sum(1 for p in new_papers if p.get('arxiv_id'))
    
    multi_source = sum(1 for p in new_papers if len(p.get('sources', [])) > 1)
    
    log_info(f"New papers analysis:")
    log_info(f"  - With abstracts: {with_abstracts} ({100*with_abstracts//len(new_papers)}%)")
    log_info(f"  - With DOIs: {with_dois} ({100*with_dois//len(new_papers)}%)")
    log_info(f"  - With OpenAlex IDs: {with_openalex}")
    log_info(f"  - With arXiv IDs: {with_arxiv}")
    log_info(f"  - From multiple sources: {multi_source}")
    
    # Show date range
    dates = [p['publication_date'] for p in new_papers if p.get('publication_date')]
    if dates:
        earliest = min(dates)
        latest = max(dates)
        log_info(f"  - Publication date range: {earliest.date()} to {latest.date()}")
    
    # Show source distribution
    source_counts = Counter()
    for paper in new_papers:
        for source in paper.get('sources', [paper.get('source', 'Unknown')]):
            source_counts[source] += 1
    log_info(f"  - Source distribution: {dict(source_counts)}")
else:
    log_info("No new papers to process after deduplication")

log_info("Deduplication and merge complete")
log_info(f"Ready to classify {len(new_papers)} papers against Research Questions")


[INFO 2026-01-19T05:43:29.653913+00:00] Starting deduplication and merge of papers from all sources...
[INFO 2026-01-19T05:43:29.654748+00:00] Total papers from all sources: 94
[INFO 2026-01-19T05:43:29.654927+00:00]   - OpenAlex: 65
[INFO 2026-01-19T05:43:29.655117+00:00]   - arXiv: 21
[INFO 2026-01-19T05:43:29.655231+00:00]   - Google Drive: 8
[INFO 2026-01-19T05:43:29.655291+00:00] Checking for duplicates against existing Notion papers...
[INFO 2026-01-19T05:43:29.658298+00:00] Found 0 papers already in Notion (will skip)
[INFO 2026-01-19T05:43:29.659316+00:00] New candidate papers after Notion deduplication: 94
[INFO 2026-01-19T05:43:29.659476+00:00] Deduplicating within new candidate papers...
[INFO 2026-01-19T05:43:29.710390+00:00] Grouped 94 candidates into 92 unique papers
[INFO 2026-01-19T05:43:29.712764+00:00] Merging 2 papers from sources: Google Drive, Google Drive
[INFO 2026-01-19T05:43:29.712816+00:00] Merging 2 papers from sources: Google Drive, Google Drive
[INFO 2026-0

In [31]:
# ============================================================
# Cell 09 — LLM classification of papers against Research Questions
# ============================================================
# Overview:
#   Uses OpenAI LLM to classify each new paper's relevance to existing
#   Research Questions, Clusters, and Gaps. Produces structured output
#   with relevance scores, matched entities, and rationale for human review.
#
# Inputs / Outputs:
#   Inputs: new_papers, classification_context, openai_client
#   Outputs: classified_papers (list with classification metadata)
#
# Notes:
#   - Batch papers to reduce API calls (e.g., 5-10 papers per prompt)
#   - Request structured JSON output with relevance scores
#   - Include paper title/abstract and RQ/Cluster/Gap summaries in prompt
#   - Handles papers without abstracts gracefully
#   - Rate limiting and error handling for API calls
#   - Classification stored but not yet persisted (done in Cell 10)

total_rqs = len(classification_context.get('research_questions', []))
total_clusters = len(classification_context.get('clusters', []))
total_gaps = len(classification_context.get('gaps', []))

if total_rqs == 0 and total_clusters == 0 and total_gaps == 0:
    raise RuntimeError(
        "Classification context is empty (0 RQs/Clusters/Gaps). "
        "Fix Notion fetch/extraction before running LLM classification."
    )

log_info("Starting LLM classification of papers against Research Questions...")

# --- Skip if no papers to classify ---
if not new_papers:
    log_info("No papers to classify")
    classified_papers = []
else:
    log_info(f"Classifying {len(new_papers)} papers...")
    
    # --- Build context summary ---
    total_rqs = len(classification_context['research_questions'])
    total_clusters = len(classification_context['clusters'])
    total_gaps = len(classification_context['gaps'])
    
    log_info(f"Classification targets: {total_rqs} RQs, {total_clusters} Clusters, {total_gaps} Gaps")
    
    if total_rqs == 0:
        log_warning("No active Research Questions available for classification")
        log_warning("All papers will be marked as unclassified")
    
    # --- Helper: Build classification context string ---
    def build_classification_context() -> str:
        """Build structured context string for LLM prompt."""
        lines = []
        
        # Research Questions
        if classification_context['research_questions']:
            lines.append("RESEARCH QUESTIONS:")
            for idx, rq in enumerate(classification_context['research_questions'][:20], 1):
                lines.append(f"{idx}. [{rq['page_id'][:8]}] {rq['title']}")
                if rq.get('description'):
                    desc = rq['description'][:150]
                    lines.append(f"   {desc}...")
            if len(classification_context['research_questions']) > 20:
                lines.append(f"   ... and {len(classification_context['research_questions']) - 20} more")
        
        # Clusters
        if classification_context['clusters']:
            lines.append("\nCLUSTERS:")
            for idx, cluster in enumerate(classification_context['clusters'][:10], 1):
                lines.append(f"{idx}. [{cluster['page_id'][:8]}] {cluster['title']}")
                if cluster.get('description'):
                    desc = cluster['description'][:100]
                    lines.append(f"   {desc}...")
            if len(classification_context['clusters']) > 10:
                lines.append(f"   ... and {len(classification_context['clusters']) - 10} more")
        
        # Gaps
        if classification_context['gaps']:
            lines.append("\nRESEARCH GAPS:")
            for idx, gap in enumerate(classification_context['gaps'][:10], 1):
                lines.append(f"{idx}. [{gap['page_id'][:8]}] {gap['title']}")
                if gap.get('description'):
                    desc = gap['description'][:100]
                    lines.append(f"   {desc}...")
            if len(classification_context['gaps']) > 10:
                lines.append(f"   ... and {len(classification_context['gaps']) - 10} more")
        
        return "\n".join(lines)
    
    # --- Helper: Format paper for LLM ---
    def format_paper_for_llm(paper: Dict[str, Any]) -> Dict[str, str]:
        """Format paper metadata for LLM prompt."""
        abstract = paper.get('abstract', '')
        if len(abstract) > 1500:
            abstract = abstract[:1500] + "..."
        
        return {
            'title': paper.get('title', 'Unknown'),
            'authors': paper.get('authors', 'Unknown'),
            'abstract': abstract if abstract else '(No abstract available)',
            'venue': paper.get('venue', 'Unknown'),
            'year': paper.get('publication_date').year if paper.get('publication_date') else 'Unknown'
        }
    
    # --- Build system prompt ---
    system_prompt = """You are a research paper classification assistant. Your task is to analyze academic papers and determine their relevance to specific Research Questions, Clusters, and Gaps.

For each paper, you must:
1. Identify which Research Questions (RQs) it directly addresses or contributes to
2. Identify which Clusters (thematic groupings) it belongs to
3. Identify which Research Gaps it helps fill
4. Provide a brief rationale for your classification
5. Assign an initial status: 'new' (clearly relevant) or 'pending' (uncertain, needs human review)

Output ONLY valid JSON matching this schema:
{
  "classifications": [
    {
      "paper_title": "<exact title>",
      "relevant_rqs": ["<rq_page_id>", ...],
      "relevant_clusters": ["<cluster_page_id>", ...],
      "relevant_gaps": ["<gap_page_id>", ...],
      "rationale": "<brief explanation>",
      "status": "new" or "pending"
    }
  ]
}

Classification guidelines:
- Only include strong matches (clear topical/methodological relevance)
- If uncertain, mark status as 'pending' rather than guessing
- Rationale should be 1-2 sentences maximum
- Empty lists are valid if no strong matches found
- Use page_id prefixes from the context (e.g., 'a1b2c3d4')"""
    
    # --- Build classification context once ---
    context_str = build_classification_context()
    
    # --- Process papers in batches ---
    BATCH_SIZE = 5
    batches = [new_papers[i:i+BATCH_SIZE] for i in range(0, len(new_papers), BATCH_SIZE)]
    
    log_info(f"Processing {len(batches)} batches (batch size: {BATCH_SIZE})")
    
    classified_papers = []
    classification_stats = {
        'total_processed': 0,
        'with_rq_matches': 0,
        'with_cluster_matches': 0,
        'with_gap_matches': 0,
        'status_new': 0,
        'status_pending': 0,
        'api_errors': 0
    }
    
    for batch_idx, batch in enumerate(batches):
        log_info(f"Classifying batch {batch_idx + 1}/{len(batches)} ({len(batch)} papers)...")
        
        try:
            # --- Build batch prompt ---
            papers_str = "\n\n".join([
                f"PAPER {i+1}:\nTitle: {format_paper_for_llm(p)['title']}\nAuthors: {format_paper_for_llm(p)['authors']}\nVenue: {format_paper_for_llm(p)['venue']} ({format_paper_for_llm(p)['year']})\nAbstract: {format_paper_for_llm(p)['abstract']}"
                for i, p in enumerate(batch)
            ])
            
            user_prompt = f"""CLASSIFICATION CONTEXT:
{context_str}

---

PAPERS TO CLASSIFY:
{papers_str}

---

Provide classifications in JSON format."""
            
            # --- Call OpenAI API ---
            response = openai_client.chat.completions.create(
                model=llm_model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=llm_temperature,
                response_format={"type": "json_object"}
            )
            
            # --- Parse response ---
            result_text = response.choices[0].message.content
            result_json = json.loads(result_text)
            
            classifications = result_json.get('classifications', [])
            
            if len(classifications) != len(batch):
                log_warning(f"Expected {len(batch)} classifications, got {len(classifications)}")
            
            # --- Attach classifications to papers ---
            for paper, classification in zip(batch, classifications):
                paper['classification'] = {
                    'relevant_rqs': classification.get('relevant_rqs', []),
                    'relevant_clusters': classification.get('relevant_clusters', []),
                    'relevant_gaps': classification.get('relevant_gaps', []),
                    'rationale': classification.get('rationale', 'No rationale provided'),
                    'status': classification.get('status', 'pending')
                }
                
                classified_papers.append(paper)
                classification_stats['total_processed'] += 1
                
                # Update stats
                if paper['classification']['relevant_rqs']:
                    classification_stats['with_rq_matches'] += 1
                if paper['classification']['relevant_clusters']:
                    classification_stats['with_cluster_matches'] += 1
                if paper['classification']['relevant_gaps']:
                    classification_stats['with_gap_matches'] += 1
                if paper['classification']['status'] == 'new':
                    classification_stats['status_new'] += 1
                else:
                    classification_stats['status_pending'] += 1
            
            log_info(f"Batch {batch_idx + 1} classified successfully")
        
        except json.JSONDecodeError as e:
            log_error(f"JSON parsing error in batch {batch_idx + 1}: {str(e)}")
            log_error(f"Raw response: {result_text[:500]}...")
            classification_stats['api_errors'] += 1
            
            # Fallback: mark papers as unclassified
            for paper in batch:
                paper['classification'] = {
                    'relevant_rqs': [],
                    'relevant_clusters': [],
                    'relevant_gaps': [],
                    'rationale': 'Classification failed (JSON parse error)',
                    'status': 'pending'
                }
                classified_papers.append(paper)
                classification_stats['total_processed'] += 1
                classification_stats['status_pending'] += 1
        
        except Exception as e:
            log_error(f"API error in batch {batch_idx + 1}: {str(e)}")
            classification_stats['api_errors'] += 1
            
            # Fallback: mark papers as unclassified
            for paper in batch:
                paper['classification'] = {
                    'relevant_rqs': [],
                    'relevant_clusters': [],
                    'relevant_gaps': [],
                    'rationale': f'Classification failed: {str(e)[:100]}',
                    'status': 'pending'
                }
                classified_papers.append(paper)
                classification_stats['total_processed'] += 1
                classification_stats['status_pending'] += 1
        
        # Rate limiting between batches
        if batch_idx < len(batches) - 1:
            time.sleep(1.0)
    
    # --- Log classification statistics ---
    log_info(f"Classification complete: {classification_stats['total_processed']} papers processed")
    log_info(f"  - With RQ matches: {classification_stats['with_rq_matches']}")
    log_info(f"  - With Cluster matches: {classification_stats['with_cluster_matches']}")
    log_info(f"  - With Gap matches: {classification_stats['with_gap_matches']}")
    log_info(f"  - Status 'new': {classification_stats['status_new']}")
    log_info(f"  - Status 'pending': {classification_stats['status_pending']}")
    
    if classification_stats['api_errors'] > 0:
        log_warning(f"API errors encountered: {classification_stats['api_errors']} batches failed")
    
    # --- Calculate match distribution ---
    if classified_papers:
        rq_match_counts = Counter()
        for paper in classified_papers:
            num_matches = len(paper['classification']['relevant_rqs'])
            rq_match_counts[num_matches] += 1
        
        log_info(f"RQ match distribution:")
        for num_matches in sorted(rq_match_counts.keys()):
            count = rq_match_counts[num_matches]
            log_info(f"  - {num_matches} RQs: {count} papers")

log_info("LLM classification complete")


[INFO 2026-01-19T05:52:25.256795+00:00] Starting LLM classification of papers against Research Questions...
[INFO 2026-01-19T05:52:25.259373+00:00] Classifying 92 papers...
[INFO 2026-01-19T05:52:25.259422+00:00] Classification targets: 30 RQs, 0 Clusters, 32 Gaps
[INFO 2026-01-19T05:52:25.259602+00:00] Processing 19 batches (batch size: 5)
[INFO 2026-01-19T05:52:25.259634+00:00] Classifying batch 1/19 (5 papers)...
[INFO 2026-01-19T05:52:33.445188+00:00] Batch 1 classified successfully
[INFO 2026-01-19T05:52:34.447005+00:00] Classifying batch 2/19 (5 papers)...
[INFO 2026-01-19T05:52:42.999803+00:00] Batch 2 classified successfully
[INFO 2026-01-19T05:52:44.004477+00:00] Classifying batch 3/19 (5 papers)...
[INFO 2026-01-19T05:52:54.386593+00:00] Batch 3 classified successfully
[INFO 2026-01-19T05:52:55.388449+00:00] Classifying batch 4/19 (5 papers)...
[INFO 2026-01-19T05:53:04.003848+00:00] Batch 4 classified successfully
[INFO 2026-01-19T05:53:05.008455+00:00] Classifying batch 5/1

In [36]:
# ============================================================
# Cell 10 — Upsert new papers to Notion (OpenAI-enriched, schema-robust)
#   - Filters to RQ-matched papers only
#   - Uses OpenAI to generate: Name / Authors & Year / Core Idea (JP) / Tags / PDF Link
#   - Merges with classic metadata + relations + status
#   - Robust to DB schema differences (title prop name, Source type, Status type, etc.)
# ============================================================

import os, json, re, time, requests
from typing import Dict, Any, List, Optional
from datetime import datetime

log_info("Upserting classified papers to Notion (OpenAI-enriched, schema-robust)...")

# ----------------------------
# Notion REST setup
# ----------------------------
NOTION_VERSION = os.getenv("NOTION_VERSION", "2022-06-28")
NOTION_HEADERS = {
    "Authorization": f"Bearer {notion_token}",
    "Notion-Version": NOTION_VERSION,
    "Content-Type": "application/json",
}

def notion_post(url: str, payload: dict, timeout: int = 30) -> requests.Response:
    return requests.post(url, headers=NOTION_HEADERS, json=payload, timeout=timeout)

def notion_patch(url: str, payload: dict, timeout: int = 30) -> requests.Response:
    return requests.patch(url, headers=NOTION_HEADERS, json=payload, timeout=timeout)

def notion_query_database(database_id: str, payload: dict, timeout: int = 30) -> dict:
    r = notion_post(f"https://api.notion.com/v1/databases/{database_id}/query", payload, timeout=timeout)
    if not r.ok:
        raise RuntimeError(f"Notion DB query failed: {r.status_code} {r.text}")
    return r.json()

def notion_get_database(database_id: str, timeout: int = 30) -> dict:
    r = requests.get(f"https://api.notion.com/v1/databases/{database_id}", headers=NOTION_HEADERS, timeout=timeout)
    if not r.ok:
        raise RuntimeError(f"Notion DB get failed: {r.status_code} {r.text}")
    return r.json()

def notion_create_page(database_id: str, properties: dict) -> dict:
    payload = {"parent": {"database_id": database_id}, "properties": properties}
    r = notion_post("https://api.notion.com/v1/pages", payload)
    if not r.ok:
        raise RuntimeError(f"Notion page create failed: {r.status_code} {r.text}")
    return r.json()

def notion_update_page(page_id: str, properties: dict) -> dict:
    payload = {"properties": properties}
    r = notion_patch(f"https://api.notion.com/v1/pages/{page_id}", payload)
    if not r.ok:
        raise RuntimeError(f"Notion page update failed: {r.status_code} {r.text}")
    return r.json()

# ----------------------------
# Property builders (safe)
# ----------------------------
def p_title(text: str) -> dict:
    text = (text or "").strip() or "Untitled Paper"
    return {"title": [{"type": "text", "text": {"content": text[:2000]}}]}

def p_rich(text: str) -> dict:
    text = (text or "").strip()
    return {"rich_text": [{"type": "text", "text": {"content": text[:2000]}}]} if text else {"rich_text": []}

def p_url(url: str) -> dict:
    url = (url or "").strip()
    return {"url": url[:2000]} if url else {"url": None}

def p_date(dt: Optional[datetime]) -> dict:
    if not dt:
        return {"date": None}
    try:
        return {"date": {"start": dt.date().isoformat()}}
    except Exception:
        return {"date": None}

def p_select(name: str) -> dict:
    name = (name or "").strip()
    return {"select": {"name": name}} if name else {"select": None}

def p_multi_select(names: List[str]) -> dict:
    cleaned = []
    for n in (names or []):
        s = (n or "").strip()
        if s:
            cleaned.append({"name": s[:100]})
    return {"multi_select": cleaned}

def p_relation(page_ids: List[str]) -> dict:
    ids = []
    for pid in (page_ids or []):
        pid = (pid or "").strip()
        if pid:
            ids.append({"id": pid})
    return {"relation": ids}

# ----------------------------
# Detect DB schema (Title property name etc.)
# ----------------------------
db = notion_get_database(NOTION_LIT_DB_ID)
db_props = db.get("properties", {}) or {}

def detect_title_prop_name(db_props: dict) -> str:
    for cand in ["Name", "Title"]:
        p = db_props.get(cand)
        if isinstance(p, dict) and p.get("type") == "title":
            return cand
    for k, p in db_props.items():
        if isinstance(p, dict) and p.get("type") == "title":
            return k
    raise RuntimeError("No title property found in Literature DB.")

TITLE_PROP = detect_title_prop_name(db_props)
log_info(f"Literature DB title property detected: {TITLE_PROP}")

# ----------------------------
# Property name mapping (edit if your DB differs)
# ----------------------------
DB_SCHEMA = {
    "authors": "Authors",
    "abstract": "Abstract",
    "doi": "DOI",
    "openalex_id": "OpenAlex ID",
    "arxiv_id": "arXiv ID",
    "venue": "Venue",
    "url": "URL",
    "publication_date": "Publication Date",
    "status": "Status",
    "source": "Source",
    "rationale": "Classification Rationale",
    "rqs": "Research Questions",
    "clusters": "Clusters",
    "gaps": "Gaps",

    # AI formatted fields you want to populate
    "ai_name": "Name",
    "ai_authors_year": "Authors & Year",
    "ai_core_idea": "Core Idea",
    "ai_tags": "Tags",
    "ai_pdf_link": "PDF Link",
}

# ----------------------------
# OpenAI -> Notion fields (Name / Authors&Year / Core Idea / Tags / PDF link)
# ----------------------------
def build_notion_fields_prompt(paper: Dict[str, Any]) -> str:
    title = (paper.get("title") or "").strip()
    authors = (paper.get("authors") or "").strip()
    abstract = (paper.get("abstract") or "").strip()
    venue = (paper.get("venue") or "").strip()
    year = paper.get("publication_date").year if paper.get("publication_date") else ""
    url = (paper.get("url") or "").strip()

    if len(abstract) > 2500:
        abstract = abstract[:2500] + "..."

    return f"""
You are preparing fields for a Notion Literature Database.

Rules:
- Do NOT hallucinate.
- Use ONLY the provided information.
- If unknown, return empty string or empty array.
- Output JSON ONLY.

Language rules:
- name: English
- authors_year: English (e.g. "Smith, John (2023)")
- core_idea: Japanese (2-3 sentences)
- tags: English, Title Case, 2–6 items
- pdf_link: URL if confidently available from URL/DOI/arXiv, else empty string

Return JSON with exactly these keys:
- name
- authors_year
- core_idea
- tags
- pdf_link

Paper information:
Title: {title}
Authors: {authors}
Venue: {venue}
Year: {year}
URL: {url}

Abstract:
{abstract}
""".strip()

def openai_generate_notion_fields(paper: Dict[str, Any]) -> Dict[str, Any]:
    system = "You are a precise research assistant. Output valid JSON only."
    user = build_notion_fields_prompt(paper)

    r = openai_client.chat.completions.create(
        model=llm_model,
        temperature=0.2,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads((r.choices[0].message.content or "").strip())

def infer_pdf_link_fallback(paper: Dict[str, Any]) -> str:
    """
    If OpenAI leaves pdf_link empty, we can safely infer some cases without hallucination:
    - arXiv: pdf_url exists or derive from arxiv_id
    - DOI: keep DOI URL (not PDF, but at least resolvable) if you want; here we prefer real PDF links only.
    """
    pdf_url = (paper.get("pdf_url") or "").strip()
    if pdf_url:
        return pdf_url

    arxiv_id = (paper.get("arxiv_id") or "").strip()
    if arxiv_id:
        return f"https://arxiv.org/pdf/{arxiv_id}.pdf"

    # If you want DOI resolver as "PDF Link" even though not a direct PDF, uncomment:
    # doi = (paper.get("doi") or "").strip()
    # if doi:
    #     return f"https://doi.org/{doi}"

    return ""

def build_notion_properties_from_ai(fields: Dict[str, Any], paper: Dict[str, Any]) -> Dict[str, Any]:
    props: Dict[str, Any] = {}

    # Name (title) — MUST use detected title prop
    # If DB title prop isn't literally "Name", we still set TITLE_PROP.
    name_prop = TITLE_PROP
    props[name_prop] = p_title(fields.get("name") or paper.get("title") or "Untitled Paper")

    # Authors & Year (rich_text)
    k = DB_SCHEMA.get("ai_authors_year")
    if k and k in db_props:
        props[k] = p_rich(fields.get("authors_year") or "")

    # Core Idea (Japanese, rich_text)
    k = DB_SCHEMA.get("ai_core_idea")
    if k and k in db_props:
        props[k] = p_rich(fields.get("core_idea") or "")

    # Tags (multi_select)
    k = DB_SCHEMA.get("ai_tags")
    if k and k in db_props:
        props[k] = p_multi_select(fields.get("tags") or [])

    # PDF Link (url)
    pdf_link = (fields.get("pdf_link") or "").strip()
    if not pdf_link:
        pdf_link = infer_pdf_link_fallback(paper)

    k = DB_SCHEMA.get("ai_pdf_link")
    if k and k in db_props:
        props[k] = p_url(pdf_link)

    return props

# ----------------------------
# Find existing page by DOI/OpenAlex/arXiv (best-effort)
# ----------------------------
def find_existing_page_id(paper: Dict[str, Any]) -> Optional[str]:
    doi = (paper.get("doi") or "").strip()
    openalex_id = (paper.get("openalex_id") or "").strip()
    arxiv_id = (paper.get("arxiv_id") or "").strip()

    def q_equals(prop: str, value: str) -> Optional[str]:
        if not value or not prop or prop not in db_props:
            return None

        pmeta = db_props.get(prop) or {}
        ptype = pmeta.get("type")

        if ptype == "rich_text":
            flt = {"property": prop, "rich_text": {"equals": value}}
        elif ptype == "title":
            flt = {"property": prop, "title": {"equals": value}}
        elif ptype == "url":
            flt = {"property": prop, "url": {"equals": value}}
        elif ptype == "select":
            flt = {"property": prop, "select": {"equals": value}}
        else:
            return None

        payload = {"page_size": 1, "filter": flt}
        res = notion_query_database(NOTION_LIT_DB_ID, payload)
        results = res.get("results") or []
        return results[0]["id"] if results else None

    pid = q_equals(DB_SCHEMA.get("doi"), doi)
    if pid:
        return pid
    pid = q_equals(DB_SCHEMA.get("openalex_id"), openalex_id)
    if pid:
        return pid
    pid = q_equals(DB_SCHEMA.get("arxiv_id"), arxiv_id)
    return pid

# ----------------------------
# Build Notion properties for a paper (classic + classification + relations)
# ----------------------------
def build_notion_properties_classic(paper: Dict[str, Any]) -> Dict[str, Any]:
    title = (paper.get("title") or "Untitled Paper").strip()
    authors = (paper.get("authors") or "").strip()
    abstract = (paper.get("abstract") or "").strip()
    doi = (paper.get("doi") or "").strip()
    openalex_id = (paper.get("openalex_id") or "").strip()
    arxiv_id = (paper.get("arxiv_id") or "").strip()
    venue = (paper.get("venue") or "").strip()
    url = (paper.get("url") or "").strip()
    pub_date = paper.get("publication_date")
    sources = paper.get("sources") or [paper.get("source")]

    if len(abstract) > 1900:
        abstract = abstract[:1900] + "..."

    classification = paper.get("classification") or {}
    status = (classification.get("status") or "pending").strip()
    rationale = (classification.get("rationale") or "").strip()

    relevant_rqs = classification.get("relevant_rqs") or []
    relevant_clusters = classification.get("relevant_clusters") or []
    relevant_gaps = classification.get("relevant_gaps") or []

    props: Dict[str, Any] = {}
    # Always set title prop to something, but AI will typically override it later anyway.
    props[TITLE_PROP] = p_title(title)

    # Classic metadata
    if DB_SCHEMA.get("authors") in db_props and authors:
        props[DB_SCHEMA["authors"]] = p_rich(authors)
    if DB_SCHEMA.get("abstract") in db_props and abstract:
        props[DB_SCHEMA["abstract"]] = p_rich(abstract)
    if DB_SCHEMA.get("doi") in db_props and doi:
        props[DB_SCHEMA["doi"]] = p_rich(doi)
    if DB_SCHEMA.get("openalex_id") in db_props and openalex_id:
        props[DB_SCHEMA["openalex_id"]] = p_rich(openalex_id)
    if DB_SCHEMA.get("arxiv_id") in db_props and arxiv_id:
        props[DB_SCHEMA["arxiv_id"]] = p_rich(arxiv_id)
    if DB_SCHEMA.get("venue") in db_props and venue:
        props[DB_SCHEMA["venue"]] = p_rich(venue)
    if DB_SCHEMA.get("url") in db_props and url:
        props[DB_SCHEMA["url"]] = p_url(url)
    if DB_SCHEMA.get("publication_date") in db_props and pub_date:
        props[DB_SCHEMA["publication_date"]] = p_date(pub_date)

    # Status (select/status/rich_text)
    status_prop = DB_SCHEMA.get("status")
    if status_prop and status_prop in db_props:
        prop_type = (db_props.get(status_prop) or {}).get("type")
        if prop_type == "select":
            props[status_prop] = p_select(status)
        elif prop_type == "status":
            props[status_prop] = {"status": {"name": status}} if status else {"status": None}
        elif prop_type == "rich_text":
            props[status_prop] = p_rich(status)

    # Source (multi_select or rich_text)
    source_prop = DB_SCHEMA.get("source")
    if source_prop and source_prop in db_props and sources:
        prop_type = (db_props.get(source_prop) or {}).get("type")
        src_list = [s for s in sources if s]
        src_text = ", ".join(src_list)
        if prop_type == "multi_select":
            props[source_prop] = p_multi_select(src_list)
        elif prop_type == "rich_text":
            props[source_prop] = p_rich(src_text)

    if DB_SCHEMA.get("rationale") in db_props and rationale:
        props[DB_SCHEMA["rationale"]] = p_rich(rationale)

    # Relations (guard IDs)
    def looks_like_notion_id(x: str) -> bool:
        x = (x or "").strip()
        return bool(re.fullmatch(r"[0-9a-fA-F]{32}", x.replace("-", ""))) or ("-" in x and len(x) >= 32)

    if DB_SCHEMA.get("rqs") in db_props:
        safe_rqs = [x for x in relevant_rqs if looks_like_notion_id(x)]
        if safe_rqs:
            props[DB_SCHEMA["rqs"]] = p_relation(safe_rqs)

    if DB_SCHEMA.get("clusters") in db_props:
        safe_clusters = [x for x in relevant_clusters if looks_like_notion_id(x)]
        if safe_clusters:
            props[DB_SCHEMA["clusters"]] = p_relation(safe_clusters)

    if DB_SCHEMA.get("gaps") in db_props:
        safe_gaps = [x for x in relevant_gaps if looks_like_notion_id(x)]
        if safe_gaps:
            props[DB_SCHEMA["gaps"]] = p_relation(safe_gaps)

    return props

# ----------------------------
# Main upsert loop
# ----------------------------
# Filter: only papers with at least 1 RQ match
rq_only = [p for p in classified_papers if (p.get("classification") or {}).get("relevant_rqs")]
log_info(f"Filtering to RQ-matched papers only: {len(rq_only)} / {len(classified_papers)}")

target_papers = rq_only

if not target_papers:
    log_info("No papers to upsert")
    upserted_page_ids = []
    upsert_stats = {"created": 0, "updated": 0, "errors": 0}
else:
    upserted_page_ids = []
    upsert_stats = {"created": 0, "updated": 0, "errors": 0}
    log_info(f"Upserting {len(target_papers)} papers... (OpenAI Notion formatting enabled)")

    for i, paper in enumerate(target_papers, 1):
        title = (paper.get("title") or "Untitled").strip()
        try:
            # 1) Classic + classification + relations
            props = build_notion_properties_classic(paper)

            # 2) OpenAI Notion formatting (cached per paper)
            ai_fields = paper.get("_notion_fields")
            if not ai_fields:
                ai_fields = openai_generate_notion_fields(paper)
                paper["_notion_fields"] = ai_fields

            # 3) Convert AI fields -> Notion props, then merge (AI overrides Name etc.)
            ai_props = build_notion_properties_from_ai(ai_fields, paper)
            props.update(ai_props)

            # Upsert by DOI/OpenAlex/arXiv
            existing_id = find_existing_page_id(paper)

            if existing_id:
                notion_update_page(existing_id, props)
                upserted_page_ids.append(existing_id)
                upsert_stats["updated"] += 1
                log_info(f"Updated [{i}/{len(target_papers)}]: {title[:80]}")
            else:
                page = notion_create_page(NOTION_LIT_DB_ID, props)
                pid = page["id"]
                upserted_page_ids.append(pid)
                upsert_stats["created"] += 1
                log_info(f"Created [{i}/{len(target_papers)}]: {title[:80]}")

            # Rate limiting: be gentle
            if i % 3 == 0:
                time.sleep(0.35)

        except Exception as e:
            upsert_stats["errors"] += 1
            log_error(f"Upsert failed [{i}/{len(target_papers)}] '{title[:60]}': {str(e)[:400]}")

    log_info(f"Upsert complete: created={upsert_stats['created']}, updated={upsert_stats['updated']}, errors={upsert_stats['errors']}")
    log_info("Papers upserted to Notion (OpenAI Notion formatting + schema-robust)")


[INFO 2026-01-19T06:20:31.184250+00:00] Upserting classified papers to Notion (OpenAI-enriched, schema-robust)...
[INFO 2026-01-19T06:20:31.889246+00:00] Literature DB title property detected: Name
[INFO 2026-01-19T06:20:31.893192+00:00] Filtering to RQ-matched papers only: 7 / 92
[INFO 2026-01-19T06:20:31.893844+00:00] Upserting 7 papers... (OpenAI Notion formatting enabled)
[INFO 2026-01-19T06:20:38.581845+00:00] Created [1/7]: LEGAL FRAMEWORK FOR BUSINESS INCUBATORS, INNOVATION CENTERS AND THEIR IMPACT ON 
[INFO 2026-01-19T06:20:45.442568+00:00] Created [2/7]: Informal economy for women entrepreneurs in developing economies: a sad economic
[INFO 2026-01-19T06:20:49.853352+00:00] Created [3/7]: Role of Alternative Investment Fund in Financing Startups
[INFO 2026-01-19T06:20:54.067861+00:00] Created [4/7]: Analisis Kontra Naratif Kebijakan Pembentukan Danantara untuk Mendukung Ketahana
[INFO 2026-01-19T06:20:58.477275+00:00] Created [5/7]: Sovereign Wealth Funds and Economic Growth in

In [40]:
# ============================================================
# Cell 11 — Update last run timestamp in Notion System Config
#   (Data Source API compatible + auto-create missing props)
# ============================================================

import os, time, json, requests
from typing import Optional, Dict, Any
from datetime import datetime, timezone

log_info("Updating last run timestamp in Notion System Config...")

# ----------------------------
# Notion REST setup
# ----------------------------
# Data Sources API is under newer versions (e.g., 2025-09-03).
# If you keep 2022-06-28, database.retrieve may still work, but you already see properties={}
NOTION_VERSION_SYS = os.getenv("NOTION_VERSION_SYS", "2025-09-03")

NOTION_HEADERS_SYS = {
    "Authorization": f"Bearer {notion_token}",
    "Notion-Version": NOTION_VERSION_SYS,
    "Content-Type": "application/json",
}

def _n_get(url: str, timeout: int = 30) -> requests.Response:
    return requests.get(url, headers=NOTION_HEADERS_SYS, timeout=timeout)

def _n_post(url: str, payload: dict, timeout: int = 30) -> requests.Response:
    return requests.post(url, headers=NOTION_HEADERS_SYS, json=payload, timeout=timeout)

def _n_patch(url: str, payload: dict, timeout: int = 30) -> requests.Response:
    return requests.patch(url, headers=NOTION_HEADERS_SYS, json=payload, timeout=timeout)

def notion_get_database_with_data_sources(database_id: str) -> dict:
    r = _n_get(f"https://api.notion.com/v1/databases/{database_id}")
    if not r.ok:
        raise RuntimeError(f"Notion database retrieve failed: {r.status_code} {r.text}")
    return r.json()

def notion_get_data_source(data_source_id: str) -> dict:
    r = _n_get(f"https://api.notion.com/v1/data_sources/{data_source_id}")
    if not r.ok:
        raise RuntimeError(f"Notion data source retrieve failed: {r.status_code} {r.text}")
    return r.json()

def notion_patch_data_source(data_source_id: str, payload: dict) -> dict:
    r = _n_patch(f"https://api.notion.com/v1/data_sources/{data_source_id}", payload)
    if not r.ok:
        raise RuntimeError(f"Notion data source update failed: {r.status_code} {r.text}")
    return r.json()

def notion_query_data_source(data_source_id: str, payload: dict) -> dict:
    # New API: POST /v1/data_sources/:data_source_id/query
    r = _n_post(f"https://api.notion.com/v1/data_sources/{data_source_id}/query", payload)
    if not r.ok:
        raise RuntimeError(f"Notion data source query failed: {r.status_code} {r.text}")
    return r.json()

def notion_create_page(database_id: str, properties: dict) -> dict:
    payload = {"parent": {"database_id": database_id}, "properties": properties}
    r = _n_post("https://api.notion.com/v1/pages", payload)
    if not r.ok:
        raise RuntimeError(f"Notion page create failed: {r.status_code} {r.text}")
    return r.json()

def notion_update_page(page_id: str, properties: dict) -> dict:
    payload = {"properties": properties}
    r = _n_patch(f"https://api.notion.com/v1/pages/{page_id}", payload)
    if not r.ok:
        raise RuntimeError(f"Notion page update failed: {r.status_code} {r.text}")
    return r.json()

# ----------------------------
# Property builders
# ----------------------------
def p_title(text: str) -> dict:
    text = (text or "").strip() or "Daily Scanner Config"
    return {"title": [{"type": "text", "text": {"content": text[:2000]}}]}

def p_date(dt: datetime) -> dict:
    # Notion accepts ISO; here we store full timestamp
    iso = dt.astimezone(timezone.utc).isoformat()
    return {"date": {"start": iso}}

# ----------------------------
# 1) Resolve Data Source ID from System Config "Database"
# ----------------------------
sys_db = notion_get_database_with_data_sources(NOTION_SYSTEM_CONFIG_DB_ID)

data_sources = sys_db.get("data_sources") or []
if not data_sources:
    raise RuntimeError(
        "System Config DB has no data_sources in API response. "
        "Check the DB ID and that the integration is shared to the DB."
    )

# If multiple, just take the first (common case)
SYS_DS_ID = data_sources[0].get("id")
if not SYS_DS_ID:
    raise RuntimeError("Could not find data_source_id under System Config DB.")

log_info(f"System Config data_source_id: {SYS_DS_ID}")

# ----------------------------
# 2) Ensure required properties exist on the Data Source schema
#     - Name: title
#     - Last Run: date
# ----------------------------
sys_ds = notion_get_data_source(SYS_DS_ID)
sys_props = sys_ds.get("properties") or {}

def has_title_prop(props: dict, name: str) -> bool:
    p = props.get(name)
    return isinstance(p, dict) and p.get("type") == "title"

def has_date_prop(props: dict, name: str) -> bool:
    p = props.get(name)
    return isinstance(p, dict) and p.get("type") == "date"

patch_props: Dict[str, Any] = {}

# Create if missing / wrong type
if not has_title_prop(sys_props, "Name"):
    patch_props["Name"] = {"type": "title", "title": {}}

if not has_date_prop(sys_props, "Last Run"):
    patch_props["Last Run"] = {"type": "date", "date": {}}

if patch_props:
    log_info(f"System Config schema missing props; updating data source schema: {list(patch_props.keys())}")
    notion_patch_data_source(SYS_DS_ID, {"properties": patch_props})
    # Refresh
    sys_ds = notion_get_data_source(SYS_DS_ID)
    sys_props = sys_ds.get("properties") or {}
    log_info("System Config schema updated.")

# ----------------------------
# 3) Upsert the single config page by Name == "Daily Scanner Config"
# ----------------------------
CONFIG_TITLE = "Daily Scanner Config"
timestamp_dt = config_state["current_timestamp"]
if not isinstance(timestamp_dt, datetime):
    timestamp_dt = datetime.now(timezone.utc)

# Find existing page
query_payload = {
    "page_size": 1,
    "filter": {
        "property": "Name",
        "title": {"equals": CONFIG_TITLE}
    }
}
res = notion_query_data_source(SYS_DS_ID, query_payload)
hits = res.get("results") or []
existing_page_id: Optional[str] = hits[0]["id"] if hits else None

properties = {
    "Name": p_title(CONFIG_TITLE),
    "Last Run": p_date(timestamp_dt),
}

if existing_page_id:
    notion_update_page(existing_page_id, properties)
    config_state["config_page_id"] = existing_page_id
    config_state["is_first_run"] = False
    log_info(f"Updated System Config page: {existing_page_id}")
else:
    page = notion_create_page(NOTION_SYSTEM_CONFIG_DB_ID, properties)
    config_state["config_page_id"] = page["id"]
    config_state["is_first_run"] = False
    log_info(f"Created System Config page: {page['id']}")

log_info(f"Last run timestamp set to: {timestamp_dt.astimezone(timezone.utc).isoformat()}")
log_info("System Config update complete")


[INFO 2026-01-19T06:30:47.990690+00:00] Updating last run timestamp in Notion System Config...
[INFO 2026-01-19T06:30:48.350069+00:00] System Config data_source_id: 2ed8e0e4-d162-80e1-892a-000bb11b5151
[INFO 2026-01-19T06:30:49.752376+00:00] Created System Config page: 2ed8e0e4-d162-819b-8b11-dfaab87a66c4
[INFO 2026-01-19T06:30:49.752822+00:00] Last run timestamp set to: 2026-01-19T05:13:09.426700+00:00
[INFO 2026-01-19T06:30:49.752964+00:00] System Config update complete


In [39]:
sys_db = notion.databases.retrieve(database_id=NOTION_SYSTEM_CONFIG_DB_ID)

print("object:", sys_db.get("object"))
print("id:", sys_db.get("id"))

sys_props = (sys_db.get("properties") or {})
print("properties count:", len(sys_props))

for k, v in sys_props.items():
    print(k, "->", v.get("type"))


object: database
id: 2ed8e0e4-d162-8095-a3ed-c1ac5f0c3a43
properties count: 0


In [41]:
# ============================================================
# Cell 12 — Generate daily review summary artifact
# ============================================================
# Overview:
#   Generates a markdown summary report of the daily scan results,
#   including statistics, new papers with classifications, and
#   prioritized review actions for human curator. Saves artifact
#   to disk with timestamp for reference.
#
# Inputs / Outputs:
#   Inputs: merge_stats, classified_papers, classification_context,
#           config_state, upsert_stats
#   Outputs: summary_report (str), summary_file_path (str)
#
# Notes:
#   - Generates human-readable markdown report
#   - Includes scan statistics, source breakdown, classification results
#   - Groups papers by relevance (high/medium/low priority)
#   - Saves to daily_review_YYYYMMDD_HHMMSS.md
#   - Keeps report concise for quick human review

log_info("Generating daily review summary artifact...")

# --- Analyze classification results ---
def count_matches(papers, key):
    return sum(len(p.get('classification', {}).get(key, [])) for p in papers)

def prioritize_papers(papers):
    """Group papers by priority based on match count."""
    high, medium, low = [], [], []
    for p in papers:
        rq_count = len(p.get('classification', {}).get('relevant_rqs', []))
        if rq_count >= 3:
            high.append(p)
        elif rq_count >= 1:
            medium.append(p)
        else:
            low.append(p)
    return high, medium, low

if classified_papers:
    high_priority, medium_priority, low_priority = prioritize_papers(classified_papers)
    total_rq_matches = count_matches(classified_papers, 'relevant_rqs')
    total_cluster_matches = count_matches(classified_papers, 'relevant_clusters')
    total_gap_matches = count_matches(classified_papers, 'relevant_gaps')
else:
    high_priority, medium_priority, low_priority = [], [], []
    total_rq_matches = total_cluster_matches = total_gap_matches = 0

# --- Build report ---
report_lines = [
    "# Daily Paper Scanner — Review Summary",
    f"**Run Date:** {config_state['current_timestamp'].strftime('%Y-%m-%d %H:%M:%S UTC')}",
    f"**Scan Period:** {config_state['last_run_timestamp'].strftime('%Y-%m-%d')} to {config_state['current_timestamp'].strftime('%Y-%m-%d')}",
    "",
    "## Scan Statistics",
    f"- Total papers scanned: {merge_stats['total_source_papers']}",
    f"  - OpenAlex: {merge_stats['openalex_papers']}",
    f"  - arXiv: {merge_stats['arxiv_papers']}",
    f"  - Google Drive: {merge_stats['gdrive_papers']}",
    f"- Existing duplicates filtered: {merge_stats['existing_duplicates']}",
    f"- New unique papers: {merge_stats['final_new_papers']}",
    f"- Papers upserted to Notion: {upsert_stats.get('created', 0)}"
]

if upsert_stats.get('errors', 0) > 0:
    report_lines.append(f"- ⚠️ Errors during upsert: {upsert_stats['errors']}")

report_lines.extend(["", "## Classification Results"])
if classified_papers:
    report_lines.extend([
        f"- Papers classified: {len(classified_papers)}",
        f"- High priority (3+ RQ matches): {len(high_priority)}",
        f"- Medium priority (1-2 RQ matches): {len(medium_priority)}",
        f"- Low priority (no RQ matches): {len(low_priority)}",
        f"- Total RQ relations: {total_rq_matches}",
        f"- Total Cluster relations: {total_cluster_matches}",
        f"- Total Gap relations: {total_gap_matches}"
    ])
else:
    report_lines.append("- No new papers classified")

report_lines.extend(["", "## New Papers for Review"])
if high_priority:
    report_lines.append("### High Priority (3+ RQ matches)")
    for p in high_priority[:10]:
        report_lines.append(f"- **{p['title'][:80]}...** ({len(p['classification']['relevant_rqs'])} RQs)")
    report_lines.append("")

if medium_priority:
    report_lines.append("### Medium Priority (1-2 RQ matches)")
    for p in medium_priority[:10]:
        report_lines.append(f"- {p['title'][:80]}... ({len(p['classification']['relevant_rqs'])} RQs)")
    report_lines.append("")

if low_priority:
    report_lines.append(f"### Low Priority / Review Later ({len(low_priority)} papers)")
    report_lines.append("")

if not classified_papers:
    report_lines.append("No new papers to review.")
    report_lines.append("")

report_lines.extend([
    "## Recommended Actions",
    "1. Review high-priority papers first in Notion",
    "2. Update Status property: new → pending/excluded/accepted",
    "3. Verify and adjust RQ/Cluster/Gap relations as needed",
    "4. Enrich metadata where incomplete (especially abstracts)",
    "5. Consider creating new RQs for papers with no matches",
    "",
    "---",
    "*Generated by 023_daily_scanner_and_incremental_ingest.ipynb*"
])

summary_report = "\n".join(report_lines)

# --- Save to file ---
timestamp_str = config_state['current_timestamp'].strftime('%Y%m%d_%H%M%S')
summary_file_path = f"daily_review_{timestamp_str}.md"

try:
    with open(summary_file_path, 'w', encoding='utf-8') as f:
        f.write(summary_report)
    log_info(f"Summary report saved to: {summary_file_path}")
except Exception as e:
    log_error(f"Error saving summary report: {str(e)}")
    summary_file_path = None

log_info(f"Summary report generated ({len(report_lines)} lines)")
log_info("Daily review summary artifact complete")


[INFO 2026-01-19T06:31:15.587382+00:00] Generating daily review summary artifact...
[INFO 2026-01-19T06:31:15.602301+00:00] Summary report saved to: daily_review_20260119_051309.md
[INFO 2026-01-19T06:31:15.602452+00:00] Summary report generated (45 lines)
[INFO 2026-01-19T06:31:15.602586+00:00] Daily review summary artifact complete


In [42]:
# ============================================================
# Cell 13 — Display summary and next actions for human review
# ============================================================
# Overview:
#   Displays the generated daily review summary in the notebook output
#   and provides clear next action steps for the human curator. Shows
#   key metrics and links to the saved summary artifact.
#
# Inputs / Outputs:
#   Inputs: summary_report (str), summary_file_path (str), merge_stats,
#           upsert_stats, classified_papers
#   Outputs: Console display of summary and action checklist
#
# Notes:
#   - Final cell in the daily scan workflow
#   - Displays concise summary for quick review
#   - Provides actionable next steps for human curator
#   - Links to saved markdown artifact for detailed review
#   - Marks end of automated scan; human review begins

log_info("Displaying daily review summary...")

# --- Display summary report ---
print("\n" + "="*80)
print("DAILY PAPER SCANNER — REVIEW SUMMARY")
print("="*80 + "\n")

if summary_report:
    print(summary_report)
else:
    log_warning("No summary report generated")

print("\n" + "="*80)

# --- Display key metrics ---
print("\n📊 KEY METRICS:")
print(f"  • New papers found: {merge_stats.get('final_new_papers', 0)}")
print(f"  • Papers added to Notion: {upsert_stats.get('created', 0)}")
print(f"  • Duplicates filtered: {merge_stats.get('existing_duplicates', 0)}")
if upsert_stats.get('errors', 0) > 0:
    print(f"  ⚠️  Errors encountered: {upsert_stats['errors']}")

# --- Display summary file location ---
if summary_file_path:
    print(f"\n📄 Detailed report saved to: {summary_file_path}")
    print(f"   Open this file for full review with paper details and priorities.")

# --- Display action checklist ---
print("\n✅ NEXT ACTIONS FOR HUMAN REVIEW:")
print("\n1. Open Notion Papers database and filter by Status = 'new'")
print("2. Review high-priority papers (those with multiple RQ matches)")
print("3. For each paper:")
print("   - Verify abstract and metadata completeness")
print("   - Confirm RQ/Cluster/Gap relations are appropriate")
print("   - Update Status: new → pending/excluded/accepted")
print("4. Consider creating new RQs for papers with no matches but clear relevance")
print("5. Enrich metadata for Google Drive PDFs (no auto-extracted abstracts)")

# --- Display time estimate ---
if classified_papers:
    estimated_minutes = len(classified_papers) * 2  # ~2 min per paper
    print(f"\n⏱️  Estimated review time: ~{estimated_minutes} minutes ({estimated_minutes//60}h {estimated_minutes%60}m)")

# --- Display workflow completion ---
print("\n" + "="*80)
print("✨ DAILY SCAN COMPLETE — AUTOMATED WORKFLOW FINISHED")
print("   Human review and curation can now begin.")
print("="*80 + "\n")

log_info("Daily review summary displayed successfully")
log_info("Notebook execution complete — ready for human review")


[INFO 2026-01-19T06:31:22.225879+00:00] Displaying daily review summary...

DAILY PAPER SCANNER — REVIEW SUMMARY

# Daily Paper Scanner — Review Summary
**Run Date:** 2026-01-19 05:13:09 UTC
**Scan Period:** 2026-01-12 to 2026-01-19

## Scan Statistics
- Total papers scanned: 94
  - OpenAlex: 65
  - arXiv: 21
  - Google Drive: 8
- Existing duplicates filtered: 0
- New unique papers: 92
- Papers upserted to Notion: 7

## Classification Results
- Papers classified: 92
- High priority (3+ RQ matches): 2
- Medium priority (1-2 RQ matches): 5
- Low priority (no RQ matches): 85
- Total RQ relations: 14
- Total Cluster relations: 7
- Total Gap relations: 8

## New Papers for Review
### High Priority (3+ RQ matches)
- **LEGAL FRAMEWORK FOR BUSINESS INCUBATORS, INNOVATION CENTERS AND THEIR IMPACT ON ...** (3 RQs)
- **Role of Alternative Investment Fund in Financing Startups...** (3 RQs)

### Medium Priority (1-2 RQ matches)
- Informal economy for women entrepreneurs in developing economies: a s